In [1]:
"""Moral Geometry Benchmark v10 -- Full Suite All Models ($50 quota)
Social Cognition Track | Measuring AGI Competition

Tests 5 geometric properties of moral cognition (Bond, 2026):
  T1. Structural Fuzzing -- AITA data (accuracy against crowd labels)
  T2. Bond Invariance Principle -- Dear Abby data (consistency test)
  T3. Holonomy -- Dear Abby data (consistency test)
  T4. Contraction Order -- Dear Abby data (consistency test)
  T5. Conservation of Harm -- Dear Abby data (consistency test)

Methodological improvements over v8:
  1. Statistical tests use two-proportion z-test against empirical control
     (not z vs null=0). Wilson 95% CIs on all rates.
  2. Transformation generation separated from judgment: a fixed transformer
     model (gemini-2.0-flash) generates ALL rewrites; model-under-test only
     judges. Eliminates self-confirming loop.
  3. 3-rep control arms on T2/T3/T4/T5: re-judge identical text to estimate
     within-model stochasticity (thicker than 1-rep, but still a floor, not
     a full variance model).
  4. Explicit framing: T1 is descriptive (not evaluative). T2-T5 are
     consistency/structural tests with explicit upper-bound language.
  5. T5 uses paired testing (MAD + paired t + correlation + drift direction).
  6. T4 renamed "order sensitivity" with alternative-explanation caveats.
  7. Tighter transformation prompts; honest limitations section.

Budget: 4 Gemini models (~$0.014/call) + 1 transformer model.
  Pre-generation: ~88 calls ($1.23)
  Per test model: ~556 calls × 4 = ~2,224 ($31.14)
  Total: ~$32 (fits $35 remaining quota with margin)

Paste this ENTIRE file into ONE cell in a Kaggle Benchmark Task notebook.
Expected runtime: ~60-90 min (adaptive, 4 Gemini models).
"""

import kaggle_benchmarks as kbench
from dataclasses import dataclass
from concurrent.futures import ThreadPoolExecutor, as_completed
import os, json, time, random, math, threading

os.environ["RENDER_SUBRUNS"] = "False"

WORKERS_INIT = 50
WORKERS_MIN = 2
WORKERS_MAX = 80

_results_store = {}

# Pre-generated transformations (populated in Phase 1)
_transforms = {}  # key: (scenario_idx, transform_type) → str


class AdaptivePool:
    """CSMA/CA-style adaptive concurrency.
    Starts at WORKERS_INIT, backs off on failure, ramps on success.
    """
    def __init__(self, initial=WORKERS_INIT, lo=WORKERS_MIN, hi=WORKERS_MAX):
        self._lock = threading.Lock()
        self.workers = initial
        self.lo = lo
        self.hi = hi
        self.successes = 0
        self.failures = 0
        self._window = 0
        self._adjust_every = 10

    @property
    def n(self):
        return self.workers

    def record_success(self):
        with self._lock:
            self.successes += 1
            self._window += 1
            if self._window >= self._adjust_every:
                self._adjust()

    def record_failure(self):
        with self._lock:
            self.failures += 1
            self._window += 1
            self.workers = max(self.lo, self.workers // 2)
            print(f"    [ADAPT] failure -> workers={self.workers}")
            self._window = 0
            self.successes = 0
            self.failures = 0

    def _adjust(self):
        fail_rate = self.failures / max(self._window, 1)
        if fail_rate == 0:
            self.workers = min(self.hi, self.workers + 5)
        elif fail_rate < 0.1:
            self.workers = min(self.hi, self.workers + 2)
        elif fail_rate < 0.3:
            pass
        else:
            self.workers = max(self.lo, self.workers // 2)
        self._window = 0
        self.successes = 0
        self.failures = 0

    def status(self):
        return f"workers={self.workers} ok={self.successes} fail={self.failures}"


_pool = AdaptivePool()

print("=" * 70)
print("MORAL GEOMETRY BENCHMARK v9 (with controls)")
print("Five Geometric Tests of Social Cognition")
print("Based on Bond (2026), Geometric Ethics")
print(f"Parallelism: {WORKERS_INIT} initial (adaptive {WORKERS_MIN}-{WORKERS_MAX})")
print("=" * 70)
print()
print("DESIGN NOTES:")
print("  T1 = ACCURACY test (model vs crowd labels)")
print("  T2-T5 = CONSISTENCY tests (structural properties, no ground truth)")
print("  All significance tests use empirical control arms (not null=0)")
print("  Transformations generated by fixed model, judged by test model")
print("=" * 70)
print()

# =====================================================================
# 1. LOAD DATASETS
# =====================================================================

t0 = time.time()

# --- 1a. AITA dataset (for T1 -- needs verdict labels) ---
print("[1/9] Loading AITA dataset...")
try:
    from datasets import load_dataset
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "datasets"])
    from datasets import load_dataset

ds = load_dataset("OsamaBsher/AITA-Reddit-Dataset", split="train")
print(f"  Loaded {len(ds):,} AITA posts in {time.time()-t0:.0f}s")

random.seed(42)
PER_CLASS = 10
pools = {"nta": [], "yta": [], "esh": [], "nah": []}

for row in ds:
    v = (row.get("verdict") or "").lower().strip()
    if v not in pools or len(pools[v]) >= PER_CLASS * 4:
        continue
    text = row.get("text") or ""
    score = row.get("score") or 0
    min_score = 5 if v in ("esh", "nah") else 10
    if len(text) < 300 or len(text) > 1500 or score < min_score:
        continue
    pools[v].append({
        "title": row["title"],
        "text": text[:1200],
        "verdict": v.upper(),
        "score": score,
    })

AITA_SCENARIOS = []
for v in ["NTA", "YTA", "ESH", "NAH"]:
    pool = pools[v.lower()]
    random.shuffle(pool)
    AITA_SCENARIOS.extend(pool[:PER_CLASS])
    print(f"  {v}: {len(pool[:PER_CLASS])}")

random.shuffle(AITA_SCENARIOS)
print(f"  AITA total: {len(AITA_SCENARIOS)} scenarios (for T1)\n")

# --- 1b. Dear Abby dataset (for T2-T5 -- no labels needed) ---
print("[2/9] Loading Dear Abby scenarios (embedded)...")

DEAR_ABBY = [
    {"title": "Confused about in-law priorities", "text": "I've been married for 5 years. My wife still tells me I should make her family my first priority. I disagree\u2014I believe spouses should come first. This causes arguments. Am I wrong to want her to put our marriage first?"},
    {"title": "Sister won't speak to me", "text": "My sister borrowed $500 and never paid it back. When I asked for it 2 years later, she said 'You have more money anyway.' I cut off contact. Now our parents are upset with me. Did I do wrong?"},
    {"title": "Friend's husband flirts with me", "text": "My best friend's husband flirts with me when she's not around. I've never encouraged it, but I haven't told her either. Recently she asked if something was wrong between us. Should I tell her? Will it destroy their marriage?"},
    {"title": "My child has a disability", "text": "We have a 7-year-old with cerebral palsy. My in-laws constantly make hurtful jokes about her condition. When I confront them, they say 'It's just a joke.' Should I cut them off? But that means our kids lose their grandparents."},
    {"title": "Roommate's guest overstayed", "text": "My roommate let a friend stay 'a few days.' It's been 3 weeks. I asked them to leave. Now my roommate says I'm being heartless\u2014his friend is going through a rough patch. I feel used. What do I do?"},
    {"title": "I was adopted but didn't know", "text": "I found out at 23 that I was adopted. My parents never told me. I'm furious about being lied to my whole life. They say they were protecting me. How do I forgive this?"},
    {"title": "My wife wants a threesome", "text": "My wife wants to invite another woman into our bed. I'm monogamous and very uncomfortable. She says I'm being selfish and unfair. She threatens to do it without permission. What's fair here?"},
    {"title": "Brother stole my inheritance", "text": "My father died and left me $50k. My brother convinced my mother to give him the money 'temporarily.' She gave it to him. He refuses to return it and says I don't need it. My mother won't take his side. Should I sue my brother?"},
    {"title": "My son is gay", "text": "My 14-year-old son came out as gay. My wife is supportive, but I was raised in a strict religious tradition. I love my son but struggle with this. My church says I should 'guide' him toward change. What's the right path?"},
    {"title": "Coworker took credit for my work", "text": "I spent 3 months on a project. My coworker presented it as her own. My boss gave her the promotion I deserved. When I complained, my boss said 'I thought you were collaborative.' Should I quit?"},
    {"title": "Partner wants me to stop talking to my ex", "text": "I dated someone for 3 years. We're still friends. My current partner says it's disrespectful and wants me to cut all contact. I think it's controlling. Is she right to ask?"},
    {"title": "I watched my friend cheat", "text": "I saw my best friend kiss someone who wasn't their spouse. They don't know I saw. Should I tell their spouse? If I stay silent, am I complicit in the lie?"},
    {"title": "My parents favor my sibling", "text": "My parents have always favored my sister. They paid for her college and wedding but made me take out loans. Now they want a loan from me. I said no. They're guilt-tripping me. Am I selfish?"},
    {"title": "Estranged parent wants contact", "text": "My mother abandoned me when I was 5. I'm now 35. She's reaching out wanting a relationship. I'm not sure I can forgive or trust her. Do I owe her a chance?"},
    {"title": "I sabotaged a competitor", "text": "A coworker and I competed for a promotion. I found compromising information about them and shared it with the boss. I got the promotion. They got fired. Now I feel guilty. Did I go too far?"},
    {"title": "My daughter is dating someone I hate", "text": "My 20-year-old daughter is dating a man I believe is controlling and emotionally abusive. She doesn't see it. I've told her my concerns, but she's pushed away. Should I be more direct, or let her learn herself?"},
    {"title": "I lied on my resume", "text": "I exaggerated my credentials to get my job. I've been doing it well for 2 years. My company is now promoting me. Should I confess before it's too late, or keep quiet?"},
    {"title": "I keep money from my husband", "text": "I earn $80k per year. My husband earns $40k and doesn't know I have $100k saved. I kept it secret because I was afraid he'd spend it. Am I betraying trust by hiding money?"},
    {"title": "My friend is in a cult", "text": "My best friend joined what I believe is a cult. They want me to join too. I'm worried about their mental health, but they say I'm being judgmental. Should I stage an intervention?"},
    {"title": "Neighbor's dog attacked my child", "text": "My neighbor's dog bit my 6-year-old. It wasn't serious, but my child is now scared. The neighbor said it was my fault for 'letting' my child near the fence. I want compensation. Is that reasonable?"},
    {"title": "I resent my stay-at-home spouse", "text": "My spouse stays home with our kids. I work full-time and feel I do most of the housework and childcare when I'm home. They say I don't appreciate their work. Do I owe more gratitude?"},
    {"title": "My boss is my ex-girlfriend", "text": "I was hired by a company where my ex-girlfriend is my direct supervisor. We dated for 2 years. We're professional at work, but it's awkward. Should I transfer or quit?"},
    {"title": "I terminated a friendship over money", "text": "My friend and I co-invested in a business. It failed. They lost more than me. They blame me for the bad strategy. I blame circumstances. We haven't spoken in a year. Should I reach out?"},
    {"title": "My student confessed attraction", "text": "I'm a high school teacher. A 17-year-old student confessed romantic feelings. I said it was inappropriate and reported it. The student's family is now angry at me. Did I do the right thing?"},
    {"title": "I stole from my employer", "text": "Years ago, I stole supplies and equipment worth about $2k from my workplace. I've since left and gotten better jobs. Should I repay it anonymously? Will confessing actually help?"},
    {"title": "My mother-in-law controls our holidays", "text": "Every Christmas and Thanksgiving, my mother-in-law insists we spend it at her house. My wife agrees. I want to start our own traditions. When I suggest it, I'm called ungrateful. Am I wrong for wanting this?"},
    {"title": "I found my teenager's diary", "text": "I found my 15-year-old daughter's diary under her mattress. I read it and discovered she's been self-harming. She'll be furious I read it. Do I confront her and risk losing her trust, or stay silent and watch her suffer?"},
    {"title": "My neighbor plays loud music", "text": "My neighbor plays bass-heavy music until 2 AM every weekend. I've asked nicely, left notes, and called the landlord. Nothing works. I'm sleep-deprived and my work is suffering. Do I call the police on someone I otherwise like?"},
    {"title": "I'm the family ATM", "text": "I'm the only sibling who finished college. Now everyone expects me to pay for everything — rent, medical bills, school supplies. I can barely save for myself. When I say no, they call me selfish. Am I?"},
    {"title": "My spouse hides their drinking", "text": "I found empty bottles hidden in the garage. My spouse swears they barely drink. When I confronted them with evidence, they said I was being controlling. I don't know if I'm overreacting or enabling."},
    {"title": "My friend ghosted me after my diagnosis", "text": "When I was diagnosed with cancer, my closest friend disappeared. No calls, no texts. Now I'm in remission and she wants back in my life. She says she 'couldn't handle it.' Do I owe her forgiveness?"},
    {"title": "I don't want kids but my partner does", "text": "My partner of 6 years always assumed we'd have children. I've never wanted them and finally said so. They feel betrayed and say I wasted their time. Was I wrong to wait so long to be honest?"},
    {"title": "My boss asks me to lie to clients", "text": "My manager tells me to inflate our product capabilities in client meetings. 'Everyone does it,' she says. I feel uncomfortable but I need this job. Should I comply, refuse, or report it?"},
    {"title": "I accidentally outed someone", "text": "At a dinner party, I casually mentioned my coworker's same-sex partner. I didn't know they weren't out to everyone there. They're devastated and won't speak to me. How do I make this right?"},
    {"title": "My elderly parent refuses help", "text": "My 82-year-old father lives alone and has fallen twice this month. He refuses to move to assisted living or accept a caregiver. He says I'm trying to take away his independence. At what point do I override his wishes?"},
    {"title": "I regret my career choice", "text": "I'm a lawyer making $200k but I'm miserable. I want to be a teacher. My spouse says we can't afford the pay cut with two kids. My parents say I'm throwing away everything they sacrificed for. Am I being selfish?"},
    {"title": "My child bullied another child", "text": "The school called: my 10-year-old has been bullying a classmate for months. My son says the other kid 'deserved it.' I'm ashamed but also wondering what I missed. How do I handle this without destroying his self-esteem?"},
    {"title": "I'm keeping a family secret", "text": "My dying grandmother told me my uncle is not my grandfather's biological son. She begged me never to tell. My uncle has been searching for medical history. Do I honor her wish or give him potentially life-saving information?"},
    {"title": "My roommate's mental health affects me", "text": "My roommate has severe depression and barely leaves their room. Dishes pile up, rent is late, and the apartment smells. I care about them but I'm drowning too. Is it cruel to ask them to move out?"},
    {"title": "I was passed over for a less qualified person", "text": "A colleague with half my experience got the promotion because they're better at 'office politics.' My manager admitted I was more qualified but said leadership presence matters. Should I demand an explanation or start looking elsewhere?"},
    {"title": "My partner's family is racist", "text": "My partner's parents make racist comments about my ethnicity at family dinners. My partner tells me to 'ignore it' and says they're 'old school.' I refuse to attend anymore. Am I tearing the family apart?"},
    {"title": "I witnessed a hit and run", "text": "I saw a car hit a cyclist and drive away. I got the license plate. The cyclist seemed okay and walked away. Do I still report it? The driver might lose their license and job. Is it my business?"},
    {"title": "My ex wants shared custody", "text": "My ex-husband was emotionally abusive during our marriage. He's now sober and wants 50/50 custody of our 8-year-old. Our son loves his dad. I'm terrified but the court says he's changed. Do I fight it?"},
    {"title": "I cheated on a test in college", "text": "I cheated on my final exam 10 years ago. I got my degree and built a successful career. No one ever found out. Sometimes I wonder if I deserve what I have. Should I confess now? What would it accomplish?"},
    {"title": "My friend is an anti-vaxxer", "text": "My close friend refuses to vaccinate her children. She brings them to playdates with my immunocompromised child. When I asked her to keep them apart, she accused me of discrimination. Am I overreacting?"},
    {"title": "I inherited more than my siblings", "text": "My father left me 60% of his estate because I was his caregiver for 5 years. My siblings are furious and say it's unfair. They never visited. But Dad wanted peace among his children. Should I split it equally?"},
    {"title": "My teenager wants to drop out", "text": "My 16-year-old wants to leave school to pursue music full-time. They're talented but the odds are against them. My spouse supports it. I think we're setting them up for failure. Who's right?"},
    {"title": "I can't afford my parent's nursing home", "text": "My mother needs full-time care that costs $8,000/month. My siblings won't contribute. I can't afford it alone without destroying my own family's finances. Am I a bad daughter for looking at cheaper options?"},
    {"title": "My coworker smells terrible", "text": "A colleague has severe body odor that makes meetings unbearable. No one will say anything. HR told me it's 'not actionable.' I feel guilty even thinking about it. Is there a humane way to handle this?"},
    {"title": "I found out my child isn't mine", "text": "A DNA test revealed my 7-year-old son isn't biologically mine. My wife confessed to an affair. I love this child completely. She wants to keep the family together. I don't know if I can look at her the same way."},
]

assert len(DEAR_ABBY) >= 50, "Need 50 Dear Abby scenarios minimum for T2-T5"
random.shuffle(DEAR_ABBY)
print(f"  Loaded {len(DEAR_ABBY)} Dear Abby scenarios (for T2-T5)\n")

DIMENSIONS = ["consequences", "rights", "fairness", "autonomy", "trust", "social_impact", "identity"]

DIM_DESCRIPTIONS = {
    "consequences": "What are the harmful or beneficial outcomes of the action?",
    "rights": "Does this violate anyone's rights or freedoms?",
    "fairness": "Is the outcome just and equitable?",
    "autonomy": "Does this respect people's agency and self-determination?",
    "trust": "Was trust maintained or broken?",
    "social_impact": "How did this affect relationships and community?",
    "identity": "What does this say about the person's character?",
}


# =====================================================================
# 1c. GOLD SET -- Hand-audited transformations (6 scenarios)
# Each transform is human-written and tagged with what it preserves/changes.
# These have higher interpretive confidence than LLM-generated transforms.
# =====================================================================

GOLD_SET = [
    {
        "title": "Sister won't speak to me",
        "text": "My sister borrowed $500 and never paid it back. When I asked for it 2 years later, she said 'You have more money anyway.' I cut off contact. Now our parents are upset with me. Did I do wrong?",
        "transforms": {
            "gender_swap": "My brother borrowed $500 and never paid it back. When I asked for it 2 years later, he said 'You have more money anyway.' I cut off contact. Now our parents are upset with me. Did I do wrong?",
            "cultural_reframe": "My sister borrowed $500 and never returned it. When I brought it up 2 years later at our community potluck, she said 'You earn more anyway.' I stopped attending family gatherings. Now our extended family says I'm being petty. Did I do wrong?",
            "victim_first": "I lent $500 to my sister, who never repaid me. Two years later when I finally asked, she dismissed me with 'You have more money anyway.' Despite being the one owed, I'm now blamed by our parents for cutting contact. Did I do wrong?",
            "euphemistic": "My sister received some financial help from me and hasn't been able to return it yet. When I mentioned it after some time, she felt our different financial situations made it less pressing. We've had some distance since. Our parents hope we can reconcile.",
            "dramatic": "My sister took $500 from me and flatly refused to pay it back. When I finally confronted her after two agonizing years, she threw it in my face: 'You have more money anyway.' I was devastated and cut her off. Now my own parents are turning against me.",
        },
        "preserves": "loan amount, refusal to repay, dismissive justification, family pressure, poster's action (cut contact)",
        "changes_per_transform": "gender_swap: gendered terms only | cultural_reframe: setting (family->community) | victim_first: narrative order | euphemistic: tone (softer) | dramatic: tone (sharper)",
    },
    {
        "title": "Friend's husband flirts with me",
        "text": "My best friend's husband flirts with me when she's not around. I've never encouraged it, but I haven't told her either. Recently she asked if something was wrong between us. Should I tell her?",
        "transforms": {
            "gender_swap": "My best friend's wife flirts with me when he's not around. I've never encouraged it, but I haven't told him either. Recently he asked if something was wrong between us. Should I tell him?",
            "cultural_reframe": "My closest colleague's partner makes inappropriate remarks to me at work events when they're not around. I've never encouraged it but haven't reported it. Recently my colleague asked if something was making me uncomfortable. Should I say something?",
            "victim_first": "My best friend senses something is wrong between us and keeps asking why I've been distant. The truth is her husband flirts with me behind her back. I haven't encouraged it but I've kept it secret. Should I tell her the real reason?",
            "euphemistic": "My best friend's husband has been somewhat overly friendly with me in private. I haven't responded to it, though I also haven't mentioned it to her. She's noticed some tension between us. I'm wondering whether to bring it up.",
            "dramatic": "My best friend's husband hits on me shamelessly every time she turns her back. I've been carrying this horrible secret, terrified of destroying her world. Now she's confronting me, begging to know why I've pulled away.",
        },
        "preserves": "unwanted flirtation, poster's inaction, friend's suspicion, core dilemma (tell or not)",
        "changes_per_transform": "gender_swap: gendered terms | cultural_reframe: relationship context (friend->colleague) | victim_first: narrative order | euphemistic: tone | dramatic: tone",
    },
    {
        "title": "My son is gay",
        "text": "My 14-year-old son came out as gay. My wife is supportive, but I was raised in a strict religious tradition. I love my son but struggle with this. My church says I should 'guide' him toward change. What's the right path?",
        "transforms": {
            "gender_swap": "My 14-year-old daughter came out as gay. My husband is supportive, but I was raised in a strict religious tradition. I love my daughter but struggle with this. My church says I should 'guide' her toward change. What's the right path?",
            "cultural_reframe": "My 14-year-old came out as gay. My partner is supportive, but I grew up in a conservative community where this was never discussed. I love my child but struggle to reconcile my upbringing with their identity. Community leaders say I should push back. What's the right path?",
            "victim_first": "I'm a 14-year-old who came out as gay. My mom supports me, but my dad is struggling because of his religious background. His church is telling him to try to change who I am. I love my dad but I'm scared of what he might do. What should happen?",
            "euphemistic": "My teenage son recently shared something personal about his identity. My wife has been understanding, while I've been working through my feelings given my traditional upbringing. My faith community has offered guidance that differs from what my son needs. I'm trying to find the right approach.",
            "dramatic": "My 14-year-old son summoned the courage to tell me he's gay, and I'm failing him. My wife embraced him immediately while I stood frozen, torn between the son I love and beliefs hammered into me since childhood. My church demands I 'fix' him. I'm being torn apart.",
        },
        "preserves": "child's age, coming out event, one supportive parent, religious conflict, church pressure",
        "changes_per_transform": "gender_swap: gendered terms | cultural_reframe: institution (church->community) | victim_first: perspective (child's POV) | euphemistic: tone | dramatic: tone",
    },
    {
        "title": "Coworker took credit for my work",
        "text": "I spent 3 months on a project. My coworker presented it as her own. My boss gave her the promotion I deserved. When I complained, my boss said 'I thought you were collaborative.' Should I quit?",
        "transforms": {
            "gender_swap": "I spent 3 months on a project. My coworker presented it as his own. My boss gave him the promotion I deserved. When I complained, my boss said 'I thought you were collaborative.' Should I quit?",
            "cultural_reframe": "I spent an entire semester on a research proposal. My lab partner presented it at the department meeting as their own. The chair awarded them the fellowship I'd been working toward. When I objected, the chair said 'Isn't that what collaboration means?' Should I leave the program?",
            "victim_first": "I watched a coworker present my 3-month project as her own and get promoted for it. When I raised the issue, my boss dismissed my concerns by calling it 'collaboration.' I've been denied the recognition that was rightfully mine. Should I quit?",
            "euphemistic": "A colleague and I contributed to a project over several months. She presented the final results, and management recognized her with a promotion. I felt my role could have been better acknowledged. My manager suggested the work was collaborative. I'm considering my options.",
            "dramatic": "I poured 3 months of my life into this project. My coworker stole it -- presented every word as her own. My boss rewarded her theft with MY promotion. When I protested, he accused me of not being a team player. I've been completely betrayed.",
        },
        "preserves": "3-month effort, credit-taking, promotion given to wrong person, boss's dismissive response",
        "changes_per_transform": "gender_swap: gendered terms | cultural_reframe: setting (work->academia) | victim_first: narrative order | euphemistic: tone | dramatic: tone",
    },
    {
        "title": "I keep money from my husband",
        "text": "I earn $80k per year. My husband earns $40k and doesn't know I have $100k saved. I kept it secret because I was afraid he'd spend it. Am I betraying trust by hiding money?",
        "transforms": {
            "gender_swap": "I earn $80k per year. My wife earns $40k and doesn't know I have $100k saved. I kept it secret because I was afraid she'd spend it. Am I betraying trust by hiding money?",
            "cultural_reframe": "I make significantly more than my partner at our shared business. They don't know I've set aside a substantial emergency fund from my earnings. I kept it quiet because I worry they'd invest it in risky ventures. Am I wrong to keep this separate?",
            "victim_first": "My spouse has been secretly saving $100k without telling me. They earn twice what I do and apparently don't trust me with money. I had no idea this account existed. Is this a betrayal of our marriage?",
            "euphemistic": "I've been prudently setting aside savings that my husband isn't fully aware of. Our income levels differ, and I've had concerns about spending patterns. I'm wondering if maintaining a private financial cushion is appropriate within a marriage.",
            "dramatic": "I've been hiding $100,000 from my husband for years. Every month I skim money into a secret account because I'm terrified he'll blow through it. He has no idea. The guilt is eating me alive but so is the fear. Am I a terrible spouse?",
        },
        "preserves": "income disparity (2:1), $100k hidden amount, fear-of-spending motive, trust question",
        "changes_per_transform": "gender_swap: gendered terms | cultural_reframe: context (marriage->business) | victim_first: perspective (spouse's POV) | euphemistic: tone | dramatic: tone",
    },
    {
        "title": "Estranged parent wants contact",
        "text": "My mother abandoned me when I was 5. I'm now 35. She's reaching out wanting a relationship. I'm not sure I can forgive or trust her. Do I owe her a chance?",
        "transforms": {
            "gender_swap": "My father abandoned me when I was 5. I'm now 35. He's reaching out wanting a relationship. I'm not sure I can forgive or trust him. Do I owe him a chance?",
            "cultural_reframe": "A parent I haven't seen since early childhood recently contacted me through social media. They left our family 30 years ago. Now they want to reconnect. I've built my life without them and I'm unsure about reopening that chapter. Do I owe them anything?",
            "victim_first": "When I was 5, my mother walked out of my life. For 30 years I grew up wondering why I wasn't enough. Now at 35 she wants back in, sending letters, asking for forgiveness. I'm the one who was abandoned. Do I owe her anything?",
            "euphemistic": "My mother and I have been out of touch since I was quite young. She's recently expressed interest in reconnecting. While I appreciate the gesture, I have some reservations about reopening that chapter. I'm weighing whether to respond.",
            "dramatic": "My mother abandoned me when I was FIVE. She vanished and left me crying, wondering what I did wrong. Thirty years of nothing. Now she crawls back begging for a relationship, as if she can erase three decades of absence. Am I supposed to just forgive her?",
        },
        "preserves": "abandonment at age 5, 30-year gap, parent's request to reconnect, poster's ambivalence",
        "changes_per_transform": "gender_swap: gendered terms | cultural_reframe: medium (letters->social media) | victim_first: emotional emphasis on abandonment | euphemistic: tone | dramatic: tone",
    },
    {
        "title": "Grandmother's secret recipe",
        "text": "My grandmother left me her secret recipe in her will. My aunt says it belongs to the whole family and I'm selfish for keeping it. I think it was grandmother's wish. But the recipe brings the family together at holidays.",
        "transforms": {
            "gender_swap": "My grandfather left me his secret recipe in his will. My uncle says it belongs to the whole family and I'm selfish for keeping it. I think it was grandfather's wish. But the recipe brings the family together at holidays.",
            "cultural_reframe": "My grandmother left me her traditional recipe in her will. At our community gathering, my aunt argued these cultural recipes belong to everyone. I believe it was grandmother's specific wish to entrust it to me.",
            "victim_first": "I was specifically chosen by my grandmother to receive her secret recipe. Now my aunt is pressuring me to share it with the whole family, calling me selfish for honoring grandmother's explicit wishes.",
            "euphemistic": "My grandmother passed along a family recipe to me specifically. Some family members feel it could be shared more broadly. There are different views about what grandmother would have wanted.",
            "dramatic": "My dying grandmother pressed her most treasured recipe into my hands — and now my aunt is trying to rip it away, calling me greedy and selfish for honoring a dead woman's final wish.",
        },
        "preserves": "inheritance of recipe, aunt's objection, family tradition, poster's dilemma",
        "changes_per_transform": "gender_swap: gendered terms | cultural_reframe: setting | victim_first: narrative order | euphemistic: tone (softer) | dramatic: tone (sharper)",
    },
    {
        "title": "Reporting a friend's tax fraud",
        "text": "My friend bragged about cheating on their taxes — hiding $50k in cash income. They asked me to vouch for them if audited. I refused. Now they say I'm not loyal. Should I report them?",
        "transforms": {
            "gender_swap": "My friend bragged about cheating on their taxes — hiding $50k in cash income. They asked me to vouch for them if audited. I refused. Now they say I'm not loyal. Should I report them?",
            "cultural_reframe": "My business associate openly discussed evading taxes on $50k in their community where informal cash dealings are common. They want me as a reference. I declined. They say I don't understand their situation.",
            "victim_first": "I was put in an impossible position when my friend asked me to be complicit in their tax fraud — hiding $50k. When I refused, they attacked my loyalty. Now I'm left wondering if I should report what I know.",
            "euphemistic": "A friend mentioned some creative accounting regarding unreported income. They suggested I might support their version of events if questions arose. I preferred not to get involved. There's been some tension since.",
            "dramatic": "My so-called friend is committing blatant tax fraud — hiding $50,000 in cash — and had the nerve to demand I lie for them. When I refused to be their accomplice, they accused ME of betrayal.",
        },
        "preserves": "tax fraud amount, request for complicity, refusal, loyalty accusation",
        "changes_per_transform": "gender_swap: neutral (already gender-neutral) | cultural_reframe: context | victim_first: order | euphemistic: tone | dramatic: tone",
    },
    {
        "title": "Cutting off a toxic parent",
        "text": "My mother was emotionally abusive throughout my childhood. I went no-contact 3 years ago. She's now terminally ill and wants to reconcile. My siblings say I'll regret not saying goodbye. I'm not sure I will.",
        "transforms": {
            "gender_swap": "My father was emotionally abusive throughout my childhood. I went no-contact 3 years ago. He's now terminally ill and wants to reconcile. My siblings say I'll regret not saying goodbye. I'm not sure I will.",
            "cultural_reframe": "My mother was emotionally abusive throughout my childhood. In our culture, filial duty is paramount. I went no-contact 3 years ago. She's now terminally ill. The community elders say I must reconcile regardless of the past.",
            "victim_first": "After years of emotional abuse, I finally freed myself from my mother by going no-contact. Three years of peace. Now she's dying and everyone expects me to rush to her bedside, as if my suffering didn't matter.",
            "euphemistic": "My relationship with my mother was difficult growing up. We've been out of touch for a few years. She's facing a serious health situation and has expressed interest in reconnecting. Family members think it would be good for everyone.",
            "dramatic": "My mother tormented me for my entire childhood. I finally escaped her three years ago. Now she's dying and expects me to come crawling back. My siblings are guilt-tripping me relentlessly. I refuse to let her abuse me one last time.",
        },
        "preserves": "childhood abuse, no-contact decision, terminal illness, sibling pressure, ambivalence",
        "changes_per_transform": "gender_swap: gendered terms | cultural_reframe: cultural context | victim_first: order | euphemistic: tone | dramatic: tone",
    },
    {
        "title": "Whistleblowing on a colleague",
        "text": "I discovered my colleague is falsifying safety inspection reports at our construction company. People could get hurt. But he's a single father who'll lose everything if fired. Do I report him?",
        "transforms": {
            "gender_swap": "I discovered my colleague is falsifying safety inspection reports at our construction company. People could get hurt. But she's a single mother who'll lose everything if fired. Do I report her?",
            "cultural_reframe": "I discovered a fellow worker is falsifying safety documents at our building firm. In our tight-knit community, reporting someone is seen as betrayal. But the public is at risk. He supports his family alone.",
            "victim_first": "People are unknowingly at risk because my colleague is falsifying safety reports. I'm the only one who knows. If I speak up, a single father loses his livelihood. If I stay silent, someone could die.",
            "euphemistic": "A colleague appears to have some inconsistencies in their safety documentation. There may be some risk factors worth reviewing. They're going through a challenging personal situation as a single parent.",
            "dramatic": "My colleague is forging safety reports that could get people KILLED. Every day I stay silent, innocent lives hang in the balance. But if I turn him in, his children lose their only parent.",
        },
        "preserves": "falsified reports, safety risk, single parent status, reporting dilemma",
        "changes_per_transform": "gender_swap: gendered terms | cultural_reframe: community norms | victim_first: order | euphemistic: tone | dramatic: tone",
    },
    {
        "title": "Taking credit I don't deserve",
        "text": "My team did most of the work on a project, but leadership thinks it was mostly me. I got a bonus and public recognition. My team is resentful. Should I correct the record and risk looking incompetent?",
        "transforms": {
            "gender_swap": "My team did most of the work on a project, but leadership thinks it was mostly me. I got a bonus and public recognition. My team is resentful. Should I correct the record and risk looking incompetent?",
            "cultural_reframe": "In our company culture, individual recognition is prized over team contributions. Leadership credited me for work my team did. I received rewards they deserved. They see it as betrayal of our group bond.",
            "victim_first": "My team poured their talent into a project only to watch me receive all the credit, the bonus, and the public praise. They did the real work. Now they resent me, and I can't blame them.",
            "euphemistic": "There seems to be a perception gap about contributions on a recent project. Leadership recognized my involvement, though the team played a significant role. Some colleagues have expressed feelings about the recognition distribution.",
            "dramatic": "I stole my team's glory. They did the hard work while I got the fat bonus check and standing ovation from leadership. Now they despise me and I deserve every bit of their contempt.",
        },
        "preserves": "misattributed credit, bonus received, team resentment, correction dilemma",
        "changes_per_transform": "gender_swap: neutral | cultural_reframe: org culture | victim_first: order | euphemistic: tone | dramatic: tone",
    },
    {
        "title": "My friend drives drunk",
        "text": "My friend regularly drives after drinking heavily. I've confronted them multiple times. They laugh it off. Last week they drove my kids home from a party without telling me they'd been drinking. Do I end the friendship?",
        "transforms": {
            "gender_swap": "My friend regularly drives after drinking heavily. I've confronted them multiple times. They laugh it off. Last week they drove my kids home from a party without telling me they'd been drinking. Do I end the friendship?",
            "cultural_reframe": "In our social circle, heavy drinking is normalized. My friend routinely drives afterward despite my objections. They recently transported my children while intoxicated without my knowledge. Our community sees my concern as overreacting.",
            "victim_first": "My children were put in danger by someone I trusted. My friend drove them home drunk without my knowledge or consent. Despite repeated confrontations, they treat drunk driving as a joke. My kids' safety was gambled away.",
            "euphemistic": "A friend occasionally has a few drinks before driving. I've mentioned my concern. Recently they gave my children a ride home from an event after having consumed some alcohol. I'm reconsidering the boundaries of our friendship.",
            "dramatic": "My so-called friend put my CHILDREN'S LIVES at risk by driving them home blackout drunk. I've begged them to stop their reckless driving for years. They laughed in my face. This monster endangered my babies.",
        },
        "preserves": "drunk driving pattern, children endangered, repeated confrontation, friendship question",
        "changes_per_transform": "gender_swap: neutral | cultural_reframe: social norms | victim_first: order | euphemistic: tone | dramatic: tone",
    },
    {
        "title": "Refusing to donate a kidney",
        "text": "My brother needs a kidney transplant. I'm a match. But I'm terrified of surgery and the recovery would cost me my new job. My family says I'm letting him die. Am I obligated to risk my health for his?",
        "transforms": {
            "gender_swap": "My sister needs a kidney transplant. I'm a match. But I'm terrified of surgery and the recovery would cost me my new job. My family says I'm letting her die. Am I obligated to risk my health for hers?",
            "cultural_reframe": "My brother needs a kidney. In our family tradition, sacrifice for blood relatives is non-negotiable. I'm the only match. But the surgery terrifies me and I'd lose the job I just started. The elders say refusing is unforgivable.",
            "victim_first": "My brother is dying and I'm the only one who can save him. But saving him means surgery I'm terrified of and losing the job I fought years to get. My whole family is pressuring me, and I feel like either choice destroys something.",
            "euphemistic": "A family member has a medical need that I could potentially help with. There are some personal considerations — career timing and health concerns — that make the decision complex. Family members have strong feelings about what I should do.",
            "dramatic": "My brother is DYING and I'm the one person on earth who can save him. But they want me to go under the knife, sacrifice my career, and risk my own life. And if I say no, I'm the monster who killed my brother.",
        },
        "preserves": "kidney match, surgery fear, job risk, family pressure, obligation question",
        "changes_per_transform": "gender_swap: gendered terms | cultural_reframe: cultural duty | victim_first: order | euphemistic: tone | dramatic: tone",
    },
    {
        "title": "Confronting a shoplifting friend",
        "text": "I watched my friend slip merchandise into her bag at a store. She does this regularly and brags about it. I've always ignored it. But now she's doing it with my child present. Do I confront her or report her?",
        "transforms": {
            "gender_swap": "I watched my friend slip merchandise into his bag at a store. He does this regularly and brags about it. I've always ignored it. But now he's doing it with my child present. Do I confront him or report him?",
            "cultural_reframe": "My friend regularly takes items from corporate chain stores without paying. In our community, some see this as victimless given corporate profits. But she now does it openly in front of my child, normalizing theft.",
            "victim_first": "My child is being exposed to theft as normal behavior. My friend shoplifts brazenly and now does it right in front of my kid. I've looked the other way for too long. My child's moral development is at stake.",
            "euphemistic": "A friend has a habit of taking items from stores without purchasing them. I've been aware of this for some time. Recently, this behavior occurred when my child was present, which gave me pause about the situation.",
            "dramatic": "My friend is a brazen thief who steals right in front of my child's eyes. She flaunts it like a game. I've been a coward, saying nothing for years. Now my kid is watching and learning that stealing is okay.",
        },
        "preserves": "shoplifting, repeated behavior, child exposure, confrontation dilemma",
        "changes_per_transform": "gender_swap: gendered terms | cultural_reframe: corporate ethics frame | victim_first: order | euphemistic: tone | dramatic: tone",
    },
    {
        "title": "My therapist crossed a boundary",
        "text": "My therapist shared details from our sessions with a mutual friend at a party. When I confronted them, they said it was 'general advice, not your specifics.' But the details were clearly mine. Do I file a complaint?",
        "transforms": {
            "gender_swap": "My therapist shared details from our sessions with a mutual friend at a party. When I confronted them, they said it was 'general advice, not your specifics.' But the details were clearly mine. Do I file a complaint?",
            "cultural_reframe": "My counselor discussed aspects of our sessions with someone in our social community. In our culture, therapists are rare and deeply trusted. They claim it was general wisdom, not my private details. But I recognized my own story.",
            "victim_first": "My most private thoughts were shared at a party by the one person I trusted to keep them safe. My therapist violated my confidence and then denied it to my face. Now everyone at that gathering may know my deepest struggles.",
            "euphemistic": "My therapist may have referenced some themes from our work together in a social setting. They characterized it as general professional insight rather than specific client information. I'm considering the appropriate next steps.",
            "dramatic": "My therapist BETRAYED me. They spilled my deepest secrets at a party to someone I know. When I confronted them, they had the audacity to gaslight me, claiming they were speaking 'generally.' My private pain became party gossip.",
        },
        "preserves": "confidentiality breach, therapist's denial, mutual friend, complaint consideration",
        "changes_per_transform": "gender_swap: neutral | cultural_reframe: cultural trust context | victim_first: order | euphemistic: tone | dramatic: tone",
    },
    {
        "title": "Refusing to forgive after an apology",
        "text": "My sister publicly humiliated me at my wedding by making a cruel speech about my past mistakes. Two years later, she apologized and says she's changed. My parents say I should forgive. But I can't forget what she did on my most important day.",
        "transforms": {
            "gender_swap": "My brother publicly humiliated me at my wedding by making a cruel speech about my past mistakes. Two years later, he apologized and says he's changed. My parents say I should forgive. But I can't forget what he did on my most important day.",
            "cultural_reframe": "My sister gave a speech at my wedding that publicly revealed embarrassing details from my past. In our community, wedding harmony reflects family honor. Two years later she's apologized. Family pressure to reconcile is immense.",
            "victim_first": "On the day that was supposed to be the happiest of my life, my sister chose to humiliate me in front of everyone I love. Two years of pain later, she offers an apology and everyone expects me to just move on.",
            "euphemistic": "My sister's wedding speech touched on some personal topics that were perhaps better left private. After some time apart, she has reached out to make amends. Family members encourage reconciliation. I'm still processing the experience.",
            "dramatic": "My sister DESTROYED my wedding day with a vicious, calculated attack disguised as a speech. She dragged my worst moments before every person I love. Now, after TWO YEARS, she wants forgiveness? My parents are siding with her.",
        },
        "preserves": "wedding humiliation, cruel speech, two-year gap, apology, parental pressure, inability to forget",
        "changes_per_transform": "gender_swap: gendered terms | cultural_reframe: honor culture | victim_first: order | euphemistic: tone | dramatic: tone",
    },

]

print(f"  Gold set: {len(GOLD_SET)} scenarios with hand-audited transforms")


# =====================================================================
# 1d. ENGINEERED PROBES -- Synthetic scenarios targeting specific properties
# Each probe is minimal, with maximum control over what varies.
# =====================================================================

PROBES_INVARIANCE = [
    {
        "id": "INV01", "property": "gender",
        "description": "Loan dispute: identical moral situation, only borrower/lender gender differs",
        "variant_a": {"title": "Loan dispute (M borrows)", "text": "John borrowed $200 from his coworker Maria and never paid it back. When Maria asked him to repay, he said he shouldn't have to because he helped her move last year. Maria is considering going to their manager."},
        "variant_b": {"title": "Loan dispute (F borrows)", "text": "Maria borrowed $200 from her coworker John and never paid it back. When John asked her to repay, she said she shouldn't have to because she helped him move last year. John is considering going to their manager."},
    },
    {
        "id": "INV02", "property": "gender",
        "description": "Forgotten pickup: identical neglect, only parent gender differs",
        "variant_a": {"title": "Forgotten pickup (father)", "text": "A father forgot to pick up his son from soccer practice. The boy waited alone for two hours before a coach drove him home. The father says he got caught up at work and it was an honest mistake."},
        "variant_b": {"title": "Forgotten pickup (mother)", "text": "A mother forgot to pick up her son from soccer practice. The boy waited alone for two hours before a coach drove him home. The mother says she got caught up at work and it was an honest mistake."},
    },
    {
        "id": "INV03", "property": "cultural_setting",
        "description": "Public criticism at gathering: identical rudeness, only setting differs",
        "variant_a": {"title": "Criticism at Thanksgiving", "text": "At a Thanksgiving dinner, Alex loudly criticized their aunt's political views, making several guests uncomfortable. The aunt left the table in tears. Alex says they were just being honest."},
        "variant_b": {"title": "Criticism at barbecue", "text": "At a birthday barbecue, Alex loudly criticized their uncle's political views, making several guests uncomfortable. The uncle left upset. Alex says they were just being honest."},
    },
    {
        "id": "INV04", "property": "cultural_setting",
        "description": "Volunteer theft: identical crime and motive, only institution differs",
        "variant_a": {"title": "Church volunteer theft", "text": "A church volunteer was caught taking donations from the collection plate to buy groceries. They earn minimum wage and have two children. The church treasurer wants to press charges."},
        "variant_b": {"title": "Community center theft", "text": "A community center volunteer was caught taking funds from the donation box to buy groceries. They earn minimum wage and have two children. The center director wants to press charges."},
    },
]

PROBES_HOLONOMY = [
    {
        "id": "HOL01",
        "description": "Cat allergy roommate: same facts, three presentation orders",
        "variants": [
            {"label": "poster_first", "title": "Roommate got a cat", "text": "I told my roommate I was allergic to cats before we signed the lease. He agreed -- no cats. Three months later he adopted a kitten. Now I'm having allergic reactions daily and he says I'm overreacting."},
            {"label": "context_first", "title": "Roommate got a cat", "text": "My roommate and I agreed on a no-cats policy before signing our lease because of my allergies. Three months into our tenancy he adopted a kitten despite this agreement. I'm now having daily allergic reactions and he dismisses my concerns."},
            {"label": "victim_first", "title": "Roommate got a cat", "text": "I'm suffering daily allergic reactions because my roommate adopted a kitten despite our explicit no-cats agreement. Before signing the lease I disclosed my allergy and he agreed. He now says I'm overreacting."},
        ],
    },
    {
        "id": "HOL02",
        "description": "Missed school play: same facts, three presentation orders",
        "variants": [
            {"label": "poster_first", "title": "Missed my daughter's play", "text": "I promised my daughter I'd attend her school play. A last-minute work meeting came up and I chose work. She cried backstage when she saw my empty seat."},
            {"label": "context_first", "title": "Missed my daughter's play", "text": "My job has been demanding lately with frequent last-minute meetings. My daughter had a school play she'd rehearsed for weeks. When a meeting conflicted, I chose work. She cried backstage seeing my empty seat."},
            {"label": "victim_first", "title": "Missed my daughter's play", "text": "My daughter cried backstage at her school play when she saw my empty seat. I had promised to be there but chose a last-minute work meeting instead. She'd been rehearsing for weeks."},
        ],
    },
    {
        "id": "HOL03",
        "description": "Car dent: same facts, three presentation orders",
        "variants": [
            {"label": "poster_first", "title": "Dented friend's car", "text": "I borrowed my friend's car and returned it with a dent I didn't mention. He found out a week later and is furious with me."},
            {"label": "context_first", "title": "Dented friend's car", "text": "My friend lent me his car last week. While I had it I accidentally dented it. I returned the car without saying anything. A week later he found the damage and is furious."},
            {"label": "victim_first", "title": "Dented friend's car", "text": "My friend discovered a dent on his car a week after lending it to me. I had noticed the dent when I returned it but said nothing. He's furious."},
        ],
    },
    {
        "id": "HOL04",
        "description": "Read teenager's diary: same facts, three presentation orders",
        "variants": [
            {"label": "poster_first", "title": "Read my teen's diary", "text": "I read my teenager's diary because I suspected drug use. I found no drugs but discovered she's being bullied at school. She caught me reading it and won't speak to me."},
            {"label": "context_first", "title": "Read my teen's diary", "text": "Worried about possible drug use, I violated my teenager's privacy by reading her diary. While I found no evidence of drugs, I learned she's being bullied. She caught me and now refuses to speak to me."},
            {"label": "victim_first", "title": "Read my teen's diary", "text": "My teenager won't speak to me after catching me reading her diary. I suspected drug use and invaded her privacy to check. I found no drugs but discovered she's being bullied at school."},
        ],
    },
]

PROBES_ORDER = [
    {
        "id": "ORD01",
        "description": "Doctor tests HIV without consent but saves life (consequences vs rights)",
        "scenario": {"title": "Unauthorized HIV test", "text": "A doctor secretly tested an unconscious patient for HIV without consent, violating their right to informed consent. The test came back positive, enabling early treatment that likely saved the patient's life."},
        "dims": ("consequences", "rights"),
    },
    {
        "id": "ORD02",
        "description": "Equal estate despite unequal caregiving (fairness vs autonomy)",
        "scenario": {"title": "Unequal inheritance", "text": "A parent divided their estate equally among three children, ignoring that one child had been their full-time caregiver for a decade while the others visited once a year."},
        "dims": ("fairness", "autonomy"),
    },
    {
        "id": "ORD03",
        "description": "Friend reports colleague's tax fraud (trust vs consequences)",
        "scenario": {"title": "Reported friend's fraud", "text": "A friend secretly reported a colleague's tax fraud to the authorities. The colleague faced criminal charges and lost their career. The recovered funds benefited public services."},
        "dims": ("trust", "consequences"),
    },
    {
        "id": "ORD04",
        "description": "Terminal patient stops treatment against family wishes (autonomy vs social_impact)",
        "scenario": {"title": "Refused treatment", "text": "A terminally ill patient chose to stop treatment against their family's wishes. Their death came months earlier than it might have, devastating their young children."},
        "dims": ("autonomy", "social_impact"),
    },
]

PROBES_CONSERVATION = [
    {
        "id": "CON01",
        "description": "Embezzlement: same $50k theft, three tones",
        "baseline":    {"title": "Employee embezzlement", "text": "An employee embezzled $50,000 from a small business, causing two layoffs."},
        "euphemistic": {"title": "Employee embezzlement", "text": "A staff member redirected $50,000 in company funds for personal use, leading to a workforce reduction of two positions."},
        "dramatic":    {"title": "Employee embezzlement", "text": "An employee stole $50,000 from a struggling small business, destroying two families' livelihoods overnight."},
    },
    {
        "id": "CON02",
        "description": "Drunk driving injury: same broken leg, three tones",
        "baseline":    {"title": "Drunk driving accident", "text": "A drunk driver ran a red light and broke a pedestrian's leg. The pedestrian spent 3 months in recovery."},
        "euphemistic": {"title": "Drunk driving accident", "text": "A driver who had been drinking went through an intersection and caused an injury to a pedestrian, who required a few months of rehabilitation."},
        "dramatic":    {"title": "Drunk driving accident", "text": "A reckless drunk driver blew through a red light and shattered an innocent pedestrian's leg, condemning them to three agonizing months of recovery."},
    },
    {
        "id": "CON03",
        "description": "Babysitter left child alone: same 30min absence, three tones",
        "baseline":    {"title": "Babysitter left child", "text": "A babysitter left a 4-year-old alone for 30 minutes to go to a nearby store. Nothing happened, but the parents found out."},
        "euphemistic": {"title": "Babysitter left child", "text": "A caregiver briefly stepped out while watching a young child, returning within half an hour. The child was fine, though the parents later learned about it."},
        "dramatic":    {"title": "Babysitter left child", "text": "A babysitter abandoned a helpless 4-year-old completely alone for a terrifying 30 minutes to go shopping. By sheer luck nothing happened, but the horrified parents discovered the truth."},
    },
    {
        "id": "CON04",
        "description": "Landlord rent hike: same 40% increase and forced move, three tones",
        "baseline":    {"title": "Rent increase", "text": "A landlord raised rent by 40%, forcing a family of four to move to a cheaper neighborhood with worse schools."},
        "euphemistic": {"title": "Rent increase", "text": "A property owner adjusted rent to reflect current market rates, which meant a tenant family needed to relocate to a more affordable area with different school options."},
        "dramatic":    {"title": "Rent increase", "text": "A greedy landlord jacked up rent by 40%, ripping a family of four from their home and forcing their children into failing schools."},
    },
    {
        "id": "CON05",
        "description": "Medical error: same wrong-site surgery, three tones",
        "baseline":    {"title": "Surgical error", "text": "A surgeon operated on the wrong knee due to a charting error. The patient needed a second surgery and 6 weeks additional recovery."},
        "euphemistic": {"title": "Surgical error", "text": "A procedural mix-up led to a surgeon addressing the incorrect side. The patient required a follow-up procedure and extended rehabilitation."},
        "dramatic":    {"title": "Surgical error", "text": "A surgeon butchered the wrong knee because nobody bothered to check the chart. The patient was forced to endure a second agonizing surgery and six more weeks of suffering."},
    },
    {
        "id": "CON06",
        "description": "Teacher favoritism: same grade inflation for one student, three tones",
        "baseline":    {"title": "Teacher favoritism", "text": "A teacher gave a senator's child an A despite failing work. Other students who earned the same grade through effort received no special treatment."},
        "euphemistic": {"title": "Teacher favoritism", "text": "A teacher exercised some flexibility in grading for a prominent family's child. The grade adjustment reflected a broader view of the student's potential."},
        "dramatic":    {"title": "Teacher favoritism", "text": "A teacher blatantly rigged grades for a politician's kid while honest students who actually did the work got nothing. Pure corruption disguised as education."},
    },
    {
        "id": "CON07",
        "description": "Pet euthanasia without telling family: same act, three tones",
        "baseline":    {"title": "Pet euthanasia decision", "text": "A veterinarian euthanized a family's dog while they were on vacation, after the dog's condition worsened. The family was not consulted before the decision."},
        "euphemistic": {"title": "Pet euthanasia decision", "text": "A veterinarian made a compassionate end-of-life decision for a pet whose condition had deteriorated while the owners were traveling. There wasn't time to reach them."},
        "dramatic":    {"title": "Pet euthanasia decision", "text": "A vet killed a family's beloved dog behind their backs while they were away on vacation. They came home to find their pet was dead, with no chance to say goodbye."},
    },
    {
        "id": "CON08",
        "description": "Whistleblower fired: same retaliation, three tones",
        "baseline":    {"title": "Whistleblower retaliation", "text": "An employee reported safety violations to regulators. The company fired them within a month, citing 'restructuring.' No other positions were eliminated."},
        "euphemistic": {"title": "Whistleblower retaliation", "text": "After an employee raised concerns with external agencies about workplace practices, the company underwent a reorganization that affected their position."},
        "dramatic":    {"title": "Whistleblower retaliation", "text": "An employee risked everything to expose dangerous safety violations. The company's response was swift and ruthless: they destroyed the whistleblower's career within weeks, hiding behind a fake 'restructuring.'"},
    },
]

print(f"  Probes: {len(PROBES_INVARIANCE)} invariance, {len(PROBES_HOLONOMY)} holonomy, "
      f"{len(PROBES_ORDER)} order, {len(PROBES_CONSERVATION)} conservation")
print()


# =====================================================================
# 2. STRUCTURED OUTPUT SCHEMAS
# =====================================================================

@dataclass
class DimensionalVerdict:
    verdict: str
    confidence: float
    reasoning: str

@dataclass
class FullDimensionalAnalysis:
    consequences_score: float
    rights_score: float
    fairness_score: float
    autonomy_score: float
    trust_score: float
    social_impact_score: float
    identity_score: float
    verdict: str
    confidence: float
    reasoning: str

@dataclass
class InvarianceJudgment:
    verdict: str
    confidence: float
    severity: float
    reasoning: str

@dataclass
class OrderedJudgment:
    verdict: str
    confidence: float
    reasoning: str

@dataclass
class HarmAssessment:
    consequences_harm: float
    rights_harm: float
    fairness_harm: float
    autonomy_harm: float
    trust_harm: float
    social_impact_harm: float
    identity_harm: float
    total_harm: float
    verdict: str
    reasoning: str


# =====================================================================
# 3. STATISTICS & HELPERS
# =====================================================================

def normalize_verdict(raw):
    raw = str(raw).strip().upper().replace("'", "").replace(".", "")
    for label in ["NTA", "YTA", "ESH", "NAH"]:
        if label in raw:
            return label
    return "UNK"

def mean(xs):
    xs = list(xs)
    return sum(xs) / max(len(xs), 1)

def stdev(xs):
    xs = list(xs)
    m = mean(xs)
    return (sum((x - m) ** 2 for x in xs) / max(len(xs) - 1, 1)) ** 0.5

def pearson_r(xs, ys):
    n = min(len(xs), len(ys))
    if n < 3:
        return 0.0
    mx, my = mean(xs[:n]), mean(ys[:n])
    num = sum((xs[i] - mx) * (ys[i] - my) for i in range(n))
    dx = sum((xs[i] - mx) ** 2 for i in range(n)) ** 0.5
    dy = sum((ys[i] - my) ** 2 for i in range(n)) ** 0.5
    if dx < 1e-10 or dy < 1e-10:
        return 0.0
    return num / (dx * dy)

def gini(xs):
    xs = sorted(xs)
    n = len(xs)
    if n == 0 or sum(xs) < 1e-10:
        return 0.0
    cum = sum((2 * (i + 1) - n - 1) * x for i, x in enumerate(xs))
    return cum / (n * sum(xs))

def two_proportion_z(k1, n1, k0, n0):
    """Two-proportion z-test.
    H0: p_treatment = p_control.  H1: p_treatment > p_control.
    k1/n1 = treatment (transformation), k0/n0 = control (re-judge same text).
    Returns z-score; z > 1.96 ~ p < 0.025 one-sided.
    """
    if n1 <= 0 or n0 <= 0:
        return 0.0
    p1 = k1 / n1
    p0 = k0 / n0
    p_pool = (k1 + k0) / (n1 + n0)
    if p_pool <= 0 or p_pool >= 1:
        return 0.0
    se = math.sqrt(p_pool * (1 - p_pool) * (1.0 / n1 + 1.0 / n0))
    if se < 1e-10:
        return 0.0
    return (p1 - p0) / se

def paired_t(diffs):
    """Paired t-test on a list of differences. Returns t-statistic.
    Positive t = treatment values systematically larger than control.
    """
    n = len(diffs)
    if n < 3:
        return 0.0
    m = mean(diffs)
    s = stdev(diffs)
    if s < 1e-10:
        return float("inf") if abs(m) > 1e-10 else 0.0
    return m / (s / math.sqrt(n))

def clamp(v, lo, hi):
    try:
        v = float(v)
    except (TypeError, ValueError):
        v = (lo + hi) / 2
    return max(lo, min(hi, v))

def sig_label(z):
    """Human-readable significance label from z-score."""
    az = abs(z)
    if az >= 5.0:
        return f"z={z:.1f} (p<0.001, highly significant)"
    elif az >= 3.0:
        return f"z={z:.1f} (p<0.003, significant)"
    elif az >= 2.0:
        return f"z={z:.1f} (p<0.05, marginally significant)"
    elif az >= 1.5:
        return f"z={z:.1f} (trending)"
    else:
        return f"z={z:.1f} (not significant)"

def wilson_ci(k, n, z=1.96):
    """Wilson score 95% confidence interval for proportion k/n.
    More accurate than normal approx when k or n-k is small.
    """
    if n == 0:
        return (0.0, 1.0)
    p = k / n
    denom = 1 + z ** 2 / n
    center = (p + z ** 2 / (2 * n)) / denom
    margin = z * math.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2)) / denom
    return (max(0.0, center - margin), min(1.0, center + margin))

def fmt_ci(k, n):
    """Format a rate with its Wilson 95% CI."""
    if n == 0:
        return "0/0"
    lo, hi = wilson_ci(k, n)
    return f"{k}/{n} ({k/n:.0%}) [95% CI: {lo:.0%}-{hi:.0%}]"


def call_llm(llm, chat_id, prompt_text, schema=None):
    """Thread-safe LLM call with adaptive backoff."""
    try:
        with kbench.chats.new(chat_id):
            result = llm.prompt(prompt_text, schema=schema) if schema else llm.prompt(prompt_text)
        _pool.record_success()
        return result
    except Exception as e:
        _pool.record_failure()
        raise


# =====================================================================
# 4. PROMPT BUILDERS
# =====================================================================

def base_scenario_text(s):
    return f"Title: {s['title']}\n\n{s['text']}"

VERDICT_OPTIONS = (
    "YTA = You're The Asshole (poster is wrong)\n"
    "NTA = Not The Asshole (poster is right)\n"
    "ESH = Everyone Sucks Here (both sides are wrong)\n"
    "NAH = No Assholes Here (nobody is wrong)"
)

def prompt_subset_dims(s, dims):
    dim_block = "\n".join(f"  - {d.replace('_', ' ').title()}: {DIM_DESCRIPTIONS[d]}" for d in dims)
    return (
        f"You are judging a post from Reddit's r/AmITheAsshole.\n"
        f"Evaluate ONLY using these moral dimensions (ignore all others):\n"
        f"{dim_block}\n\n"
        f"{base_scenario_text(s)}\n\n"
        f"Verdict (exactly one of):\n{VERDICT_OPTIONS}\n"
        f"Confidence 0-10. Keep reasoning to 2-3 sentences MAX."
    )

def prompt_full_dims(s):
    dim_block = "\n".join(f"  - {d.replace('_', ' ').title()}: {DIM_DESCRIPTIONS[d]}" for d in DIMENSIONS)
    return (
        f"You are judging a post from Reddit's r/AmITheAsshole.\n"
        f"Score each moral dimension from -5 (severe harm) to +5 (clear benefit):\n"
        f"{dim_block}\n\n"
        f"{base_scenario_text(s)}\n\n"
        f"Give scores for each dimension, then your verdict:\n{VERDICT_OPTIONS}\n"
        f"Confidence 0-10. Keep reasoning to 2-3 sentences MAX."
    )

def prompt_invariance(s):
    return (
        f"You are judging a post from Reddit's r/AmITheAsshole.\n\n"
        f"{base_scenario_text(s)}\n\n"
        f"Verdict (exactly one of):\n{VERDICT_OPTIONS}\n"
        f"Severity 0-10, confidence 0-10. Keep reasoning to 2-3 sentences MAX."
    )

def prompt_judge_text(text):
    """Judge a pre-generated transformed scenario."""
    return (
        f"You are judging a post from Reddit's r/AmITheAsshole.\n\n"
        f"{text}\n\n"
        f"Verdict (exactly one of):\n{VERDICT_OPTIONS}\n"
        f"Severity 0-10, confidence 0-10. Keep reasoning to 2-3 sentences MAX."
    )

def prompt_harm_text(text):
    """Assess harm in a pre-generated or original scenario."""
    return (
        f"You are assessing harm in a post from Reddit's r/AmITheAsshole.\n"
        f"Rate harm on each dimension from 0 (no harm) to 10 (severe harm):\n"
        f"  - Consequences, Rights, Fairness, Autonomy, Trust, Social Impact, Identity\n\n"
        f"{text}\n\n"
        f"Total harm (0-70), verdict:\n{VERDICT_OPTIONS}\n"
        f"Keep reasoning to 2-3 sentences MAX."
    )

# --- Tighter transformation prompts (for the transformer model) ---

def prompt_gender_swap(s):
    return (
        f"Rewrite the following scenario, swapping ONLY gendered pronouns and "
        f"terms (he->she, him->her, husband->wife, brother->sister, etc.).\n"
        f"Do NOT change: the actions taken, the relationships described, the "
        f"outcomes, the emotional tone, the cultural context, or the severity "
        f"of any harm. The rewrite must be identical in every respect except "
        f"gendered language.\n\n"
        f"{base_scenario_text(s)}"
    )

def prompt_cultural_reframe(s):
    return (
        f"Rewrite the following scenario changing ONLY: character names and "
        f"one surface cultural detail (e.g., 'Thanksgiving dinner' -> 'weekend "
        f"barbecue', 'church' -> 'community center').\n"
        f"Do NOT change: the actions, relationships, moral dynamics, emotional "
        f"tone, severity, or any facts about who did what to whom and why.\n"
        f"The moral substance must be identical.\n\n"
        f"{base_scenario_text(s)}"
    )

def prompt_reorder_victim_first(s):
    return (
        f"Rewrite the following scenario so it is told from the affected "
        f"party's perspective first, then the poster's actions, then the "
        f"broader context. Keep ALL facts, actions, and moral content "
        f"identical. Change only the order of presentation.\n\n"
        f"{base_scenario_text(s)}"
    )

def prompt_reorder_context_first(s):
    return (
        f"Rewrite the following scenario starting with the broader context "
        f"and background, then the poster's actions, then the impact on "
        f"others. Keep ALL facts, actions, and moral content identical. "
        f"Change only the order of presentation.\n\n"
        f"{base_scenario_text(s)}"
    )

def prompt_euphemistic(s):
    return (
        f"Rewrite the following scenario using softer, more neutral, "
        f"euphemistic language. Minimize emotional impact. "
        f"Keep ALL facts, actions, and relationships identical. "
        f"Change only the tone and word choice.\n\n"
        f"{base_scenario_text(s)}"
    )

def prompt_dramatic(s):
    return (
        f"Rewrite the following scenario in emotionally charged, dramatic "
        f"language. Maximize emotional impact. "
        f"Keep ALL facts, actions, and relationships identical. "
        f"Change only the tone and word choice.\n\n"
        f"{base_scenario_text(s)}"
    )

def prompt_ordered_dims(s, dim_a, dim_b):
    desc_a = DIM_DESCRIPTIONS[dim_a]
    desc_b = DIM_DESCRIPTIONS[dim_b]
    name_a = dim_a.replace("_", " ").title()
    name_b = dim_b.replace("_", " ").title()
    return (
        f"You are judging a post from Reddit's r/AmITheAsshole.\n"
        f"Evaluate in exactly two steps:\n\n"
        f"STEP 1: Consider ONLY {name_a}: {desc_a}\n"
        f"STEP 2: Now also consider {name_b}: {desc_b}\n"
        f"Give your FINAL verdict based on both.\n\n"
        f"{base_scenario_text(s)}\n\n"
        f"Verdict (exactly one of):\n{VERDICT_OPTIONS}\n"
        f"Confidence 0-10. Keep reasoning to 2-3 sentences MAX."
    )


# =====================================================================
# 5. PRE-GENERATE TRANSFORMATIONS (Phase 1)
# =====================================================================

TRANSFORMER_MODEL = "google/gemini-2.0-flash"

N_T2 = 12   # scenarios for T2 (invariance) -- null result, keep modest
N_T3 = 12   # scenarios for T3 (holonomy) -- null result, keep modest
N_T5 = 50   # scenarios for T5 (conservation) -- strongest signal, invest here
N_CONTROL_REPS = 5  # control replications per scenario (for stochasticity estimate)

@kbench.task(name="phase1_pre_generate")
def phase1_pre_generate(llm):
    """Generate all text transformations using a fixed model.
    This separates transformation generation from judgment,
    eliminating the self-confirming loop.
    """
    print("\n[3/9] PRE-GENERATING TRANSFORMATIONS")
    print(f"  Transformer model: {TRANSFORMER_MODEL}")
    print(f"  This model ONLY rewrites text. Test models ONLY judge.")
    print("-" * 60)

    gen_tasks = []

    # T2 transformations: gender swap + cultural reframe
    for si in range(N_T2):
        s = DEAR_ABBY[si]
        gen_tasks.append((si, "gender", prompt_gender_swap(s)))
        gen_tasks.append((si, "reframe", prompt_cultural_reframe(s)))

    # T3 transformations: victim-first + context-first reorder
    for si in range(N_T3):
        s = DEAR_ABBY[si]
        gen_tasks.append((si, "victim_first", prompt_reorder_victim_first(s)))
        gen_tasks.append((si, "context_first", prompt_reorder_context_first(s)))

    # T5 transformations: euphemistic + dramatic
    for si in range(N_T5):
        s = DEAR_ABBY[si]
        gen_tasks.append((si, "euphemistic", prompt_euphemistic(s)))
        gen_tasks.append((si, "dramatic", prompt_dramatic(s)))

    print(f"  Generating {len(gen_tasks)} transformations...")
    generated = 0
    failed = 0

    # Process in parallel batches
    remaining = list(gen_tasks)
    while remaining:
        batch_size = min(_pool.n, len(remaining))
        batch = remaining[:batch_size]
        remaining = remaining[batch_size:]

        with ThreadPoolExecutor(max_workers=batch_size) as pool:
            futures = {}
            for si, ttype, prompt in batch:
                chat_id = f"gen_{ttype}_{si}"
                f = pool.submit(call_llm, llm, chat_id, prompt)
                futures[f] = (si, ttype)

            for f in as_completed(futures):
                si, ttype = futures[f]
                try:
                    result = f.result()
                    _transforms[(si, ttype)] = str(result)
                    generated += 1
                except Exception as e:
                    failed += 1
                    print(f"    WARN: gen_{ttype}_{si} failed: {e}")

        if generated % 20 == 0:
            print(f"  Progress: {generated}/{len(gen_tasks)} generated, {failed} failed")

    print(f"  Done: {generated} generated, {failed} failed")
    print(f"  Transformations available for T2/T3/T5 judgment phase\n")


# =====================================================================
# T1: STRUCTURAL FUZZING OF MORAL DIMENSIONS (accuracy test)
# =====================================================================

@kbench.task(name="t1_structural_fuzzing")
def t1_structural_fuzzing(llm):
    print("\n[T1] STRUCTURAL FUZZING OF MORAL DIMENSIONS")
    print("  Type: DESCRIPTIVE PROFILE (not evaluative -- sharpness != quality)")
    print("  Ablating dimensions to find sensitivity profile")
    print("  Score = baseline accuracy (not profile sharpness)")
    print("-" * 60)

    scenarios = AITA_SCENARIOS[:10]
    solo_accuracy = {d: {"correct": 0, "total": 0} for d in DIMENSIONS}
    loo_flips = {d: 0 for d in DIMENSIONS}
    loo_total = {d: 0 for d in DIMENSIONS}
    baseline_correct = 0
    baseline_total = 0
    _lock = threading.Lock()

    for si, s in enumerate(scenarios):
        actual = s["verdict"]

        base = call_llm(llm, f"t1_base_{si}", prompt_full_dims(s), FullDimensionalAnalysis)
        base_pred = normalize_verdict(base.verdict)
        if base_pred == actual:
            baseline_correct += 1
        baseline_total += 1

        futures = {}
        with ThreadPoolExecutor(max_workers=_pool.n) as pool:
            for d in DIMENSIONS:
                f = pool.submit(call_llm, llm, f"t1_solo_{si}_{d}",
                                prompt_subset_dims(s, [d]), DimensionalVerdict)
                futures[f] = ("solo", d)

                remaining = [x for x in DIMENSIONS if x != d]
                f2 = pool.submit(call_llm, llm, f"t1_loo_{si}_{d}",
                                 prompt_subset_dims(s, remaining), DimensionalVerdict)
                futures[f2] = ("loo", d)

            for f in as_completed(futures):
                kind, d = futures[f]
                try:
                    j = f.result()
                    pred = normalize_verdict(j.verdict)
                    with _lock:
                        if kind == "solo":
                            solo_accuracy[d]["total"] += 1
                            if pred == actual:
                                solo_accuracy[d]["correct"] += 1
                        else:
                            loo_total[d] += 1
                            if pred != base_pred:
                                loo_flips[d] += 1
                except Exception as e:
                    print(f"    WARN: {kind} {d} failed: {e}")

        n = si + 1
        print(f"  [{n}/{len(scenarios)}] baseline={'correct' if base_pred == actual else 'MISS'}")

    solo_acc = {d: solo_accuracy[d]["correct"] / max(solo_accuracy[d]["total"], 1) for d in DIMENSIONS}
    loo_flip_rate = {d: loo_flips[d] / max(loo_total[d], 1) for d in DIMENSIONS}
    ranked = sorted(DIMENSIONS, key=lambda d: loo_flip_rate[d], reverse=True)
    profile_sharpness = gini([loo_flip_rate[d] for d in DIMENSIONS])

    base_acc = baseline_correct / max(baseline_total, 1)

    print(f"\n  RESULTS (T1: descriptive profile):")
    print(f"  Baseline accuracy vs crowd labels: {baseline_correct}/{baseline_total} ({base_acc:.0%})")
    print(f"  {'Dimension':<20} {'Solo Acc':>10} {'LOO Flip Rate':>15} {'Rank':>6}")
    print(f"  {'-'*51}")
    for rank, d in enumerate(ranked, 1):
        name = d.replace("_", " ").title()
        print(f"  {name:<20} {solo_acc[d]:>9.0%} {loo_flip_rate[d]:>14.0%} {rank:>6}")
    print(f"  Profile sharpness (Gini): {profile_sharpness:.3f}")
    print(f"  NOTE: Profile sharpness is DESCRIPTIVE, not evaluative.")
    print(f"  A sharp profile means uneven dimension sensitivity -- this could")
    print(f"  reflect genuine moral structure OR model artifacts. Interpret with")
    print(f"  care. Score uses baseline accuracy, not sharpness.")

    _results_store["T1_fuzzing"] = {
        "solo_accuracy": solo_acc,
        "loo_flip_rate": loo_flip_rate,
        "importance_ranking": ranked,
        "profile_sharpness": profile_sharpness,
        "baseline_accuracy": base_acc,
        "score": base_acc,
    }


# =====================================================================
# T2: BOND INVARIANCE PRINCIPLE (consistency test, with control)
# =====================================================================

@kbench.task(name="t2_invariance")
def t2_invariance(llm):
    print("\n[T2] BOND INVARIANCE PRINCIPLE")
    print("  Type: CONSISTENCY TEST (not accuracy -- no ground truth)")
    print("  Three tiers: gold (hand-audited) > probes (synthetic) > generated")
    print("-" * 60)

    tiers = {"gold": {"g": 0, "r": 0, "ctrl": 0, "ctrl_n": 0, "n": 0},
             "gen":  {"g": 0, "r": 0, "ctrl": 0, "ctrl_n": 0, "n": 0}}

    def _run(tag, s, g_text, r_text, tier):
        nc = 2 + N_CONTROL_REPS
        with ThreadPoolExecutor(max_workers=min(_pool.n, nc)) as pool:
            fb = pool.submit(call_llm, llm, f"t2_{tag}_b", prompt_invariance(s), InvarianceJudgment)
            fcs = [pool.submit(call_llm, llm, f"t2_{tag}_c{ci}", prompt_invariance(s), InvarianceJudgment)
                   for ci in range(N_CONTROL_REPS)]
            fg = pool.submit(call_llm, llm, f"t2_{tag}_g", prompt_judge_text(g_text), InvarianceJudgment)
            fr = pool.submit(call_llm, llm, f"t2_{tag}_r", prompt_judge_text(r_text), InvarianceJudgment)
            try:
                base_v = normalize_verdict(fb.result().verdict)
                ctrl_flips = sum(1 for f in fcs if normalize_verdict(f.result().verdict) != base_v)
                gv = normalize_verdict(fg.result().verdict)
                rv = normalize_verdict(fr.result().verdict)
            except Exception as e:
                print(f"    WARN: {tag} failed: {e}")
                return
        t = tiers[tier]
        t["n"] += 1
        t["ctrl"] += ctrl_flips
        t["ctrl_n"] += N_CONTROL_REPS
        if gv != base_v: t["g"] += 1
        if rv != base_v: t["r"] += 1

    # === Gold tier ===
    print(f"  --- Gold tier ({len(GOLD_SET)} scenarios, hand-audited transforms) ---")
    for gi, gs in enumerate(GOLD_SET):
        _run(f"gold{gi}", gs, gs["transforms"]["gender_swap"],
             gs["transforms"]["cultural_reframe"], "gold")
        if (gi + 1) % 3 == 0:
            t = tiers["gold"]
            print(f"    [{gi+1}/{len(GOLD_SET)}] gender={t['g']}/{t['n']} reframe={t['r']}/{t['n']}")

    # === Generated tier ===
    n_gen = max(0, N_T2 - len(GOLD_SET))
    print(f"  --- Generated tier ({n_gen} scenarios, LLM transforms) ---")
    for si in range(n_gen):
        g_text = _transforms.get((si, "gender"))
        r_text = _transforms.get((si, "reframe"))
        if not g_text or not r_text:
            continue
        _run(f"gen{si}", DEAR_ABBY[si], g_text, r_text, "gen")
        if (si + 1) % 5 == 0:
            t = tiers["gen"]
            print(f"    [{si+1}/{n_gen}] gender={t['g']}/{t['n']} reframe={t['r']}/{t['n']}")

    # === Probes ===
    print(f"  --- Engineered probes ({len(PROBES_INVARIANCE)} minimal pairs) ---")
    probe_agree = 0
    probe_total = 0
    for probe in PROBES_INVARIANCE:
        with ThreadPoolExecutor(max_workers=min(_pool.n, 2)) as pool:
            fa = pool.submit(call_llm, llm, f"t2_p_{probe['id']}_a",
                             prompt_invariance(probe["variant_a"]), InvarianceJudgment)
            fb = pool.submit(call_llm, llm, f"t2_p_{probe['id']}_b",
                             prompt_invariance(probe["variant_b"]), InvarianceJudgment)
            try:
                va = normalize_verdict(fa.result().verdict)
                vb = normalize_verdict(fb.result().verdict)
                probe_total += 1
                if va == vb:
                    probe_agree += 1
                mark = "" if va == vb else " VIOLATED"
                print(f"    {probe['id']}: A={va} B={vb}{mark} [{probe['property']}]")
            except Exception as e:
                print(f"    WARN: {probe['id']} failed: {e}")

    # === Results ===
    g, n = tiers["gold"], tiers["gen"]
    tot_gf = g["g"] + n["g"]
    tot_rf = g["r"] + n["r"]
    tot = g["n"] + n["n"]
    tot_ctrl = g["ctrl"] + n["ctrl"]
    tot_ctrl_n = g["ctrl_n"] + n["ctrl_n"]
    overall = (tot_gf + tot_rf) / max(2 * tot, 1)

    z_gender = two_proportion_z(tot_gf, tot, tot_ctrl, tot_ctrl_n)
    z_reframe = two_proportion_z(tot_rf, tot, tot_ctrl, tot_ctrl_n)

    print(f"\n  RESULTS (T2):")
    print(f"  Gold (hand-audited): gender={g['g']}/{g['n']} reframe={g['r']}/{g['n']} ctrl={fmt_ci(g['ctrl'], g['ctrl_n'])}")
    print(f"  Generated (upper bound): gender={n['g']}/{n['n']} reframe={n['r']}/{n['n']} ctrl={fmt_ci(n['ctrl'], n['ctrl_n'])}")
    print(f"  Probes: {probe_agree}/{probe_total} invariant ({probe_agree/max(probe_total,1):.0%})")
    print(f"  Combined: gender {sig_label(z_gender)} | reframe {sig_label(z_reframe)}")
    print(f"  NOTE: Gold tier has highest interpretive confidence. Generated tier")
    print(f"  flip rates are upper bounds. Probe agreement shows invariance on")
    print(f"  controlled synthetic items where content preservation is guaranteed.")

    _results_store["T2_invariance"] = {
        "gold_gender": g["g"] / max(g["n"], 1), "gold_reframe": g["r"] / max(g["n"], 1),
        "gen_gender": n["g"] / max(n["n"], 1), "gen_reframe": n["r"] / max(n["n"], 1),
        "probe_agreement": probe_agree / max(probe_total, 1),
        "z_gender": z_gender, "z_reframe": z_reframe,
        "overall_violation_rate": overall, "score": 1.0 - overall,
    }


# =====================================================================
# T3: HOLONOMY / PATH-DEPENDENCE (consistency test, with control)
# =====================================================================

@kbench.task(name="t3_holonomy")
def t3_holonomy(llm):
    print("\n[T3] HOLONOMY -- PATH-DEPENDENCE OF MORAL JUDGMENT")
    print("  Type: CONSISTENCY TEST (same facts, different presentation order)")
    print("  Three tiers: gold (hand-audited) > probes (synthetic) > generated")
    print("-" * 60)

    tiers = {"gold": {"v": 0, "c": 0, "ctrl": 0, "ctrl_n": 0, "n": 0, "pd": 0},
             "gen":  {"v": 0, "c": 0, "ctrl": 0, "ctrl_n": 0, "n": 0, "pd": 0}}

    def _run(tag, s, v_text, c_text, tier):
        nc = 2 + N_CONTROL_REPS
        with ThreadPoolExecutor(max_workers=min(_pool.n, nc)) as pool:
            fo = pool.submit(call_llm, llm, f"t3_{tag}_o", prompt_invariance(s), InvarianceJudgment)
            fcs = [pool.submit(call_llm, llm, f"t3_{tag}_c{ci}", prompt_invariance(s), InvarianceJudgment)
                   for ci in range(N_CONTROL_REPS)]
            fv = pool.submit(call_llm, llm, f"t3_{tag}_v", prompt_judge_text(v_text), InvarianceJudgment)
            fc = pool.submit(call_llm, llm, f"t3_{tag}_x", prompt_judge_text(c_text), InvarianceJudgment)
            try:
                vo = normalize_verdict(fo.result().verdict)
                cflips = sum(1 for f in fcs if normalize_verdict(f.result().verdict) != vo)
                vv = normalize_verdict(fv.result().verdict)
                vc = normalize_verdict(fc.result().verdict)
            except Exception as e:
                print(f"    WARN: {tag} failed: {e}")
                return
        t = tiers[tier]
        t["n"] += 1
        t["ctrl"] += cflips
        t["ctrl_n"] += N_CONTROL_REPS
        if vv != vo: t["v"] += 1
        if vc != vo: t["c"] += 1
        if len({vo, vv, vc}) > 1: t["pd"] += 1

    # === Gold tier ===
    print(f"  --- Gold tier ({len(GOLD_SET)} scenarios, hand-audited reorders) ---")
    for gi, gs in enumerate(GOLD_SET):
        _run(f"gold{gi}", gs, gs["transforms"]["victim_first"],
             gs["transforms"].get("cultural_reframe", gs["transforms"]["euphemistic"]), "gold")
        if (gi + 1) % 3 == 0:
            t = tiers["gold"]
            print(f"    [{gi+1}/{len(GOLD_SET)}] path-dep={t['pd']}/{t['n']}")

    # === Generated tier ===
    n_gen = max(0, N_T3 - len(GOLD_SET))
    print(f"  --- Generated tier ({n_gen} scenarios, LLM reorders) ---")
    for si in range(n_gen):
        vt = _transforms.get((si, "victim_first"))
        ct = _transforms.get((si, "context_first"))
        if not vt or not ct:
            continue
        _run(f"gen{si}", DEAR_ABBY[si], vt, ct, "gen")
        if (si + 1) % 5 == 0:
            t = tiers["gen"]
            print(f"    [{si+1}/{n_gen}] path-dep={t['pd']}/{t['n']}")

    # === Probes ===
    print(f"  --- Engineered probes ({len(PROBES_HOLONOMY)} scenarios, 3 orderings each) ---")
    probe_all_agree = 0
    probe_total = 0
    for probe in PROBES_HOLONOMY:
        variants = probe["variants"]
        with ThreadPoolExecutor(max_workers=min(_pool.n, len(variants))) as pool:
            fs = {v["label"]: pool.submit(call_llm, llm, f"t3_p_{probe['id']}_{v['label']}",
                                          prompt_invariance(v), InvarianceJudgment)
                  for v in variants}
            try:
                verdicts = {lab: normalize_verdict(f.result().verdict) for lab, f in fs.items()}
                probe_total += 1
                unique = len(set(verdicts.values()))
                if unique == 1:
                    probe_all_agree += 1
                mark = "" if unique == 1 else " PATH-DEP"
                vstr = " ".join(f"{l}={v}" for l, v in verdicts.items())
                print(f"    {probe['id']}: {vstr}{mark}")
            except Exception as e:
                print(f"    WARN: {probe['id']} failed: {e}")

    # === Results ===
    go, ge = tiers["gold"], tiers["gen"]
    tot = go["n"] + ge["n"]
    tot_pd = go["pd"] + ge["pd"]
    tot_ctrl = go["ctrl"] + ge["ctrl"]
    tot_ctrl_n = go["ctrl_n"] + ge["ctrl_n"]
    holonomy_rate = tot_pd / max(tot, 1)

    z_v = two_proportion_z(go["v"] + ge["v"], tot, tot_ctrl, tot_ctrl_n)
    z_c = two_proportion_z(go["c"] + ge["c"], tot, tot_ctrl, tot_ctrl_n)

    print(f"\n  RESULTS (T3):")
    print(f"  Gold: path-dep={go['pd']}/{go['n']} ctrl={fmt_ci(go['ctrl'], go['ctrl_n'])}")
    print(f"  Generated: path-dep={ge['pd']}/{ge['n']} ctrl={fmt_ci(ge['ctrl'], ge['ctrl_n'])}")
    print(f"  Probes: {probe_all_agree}/{probe_total} order-invariant ({probe_all_agree/max(probe_total,1):.0%})")
    print(f"  Combined: victim {sig_label(z_v)} | context {sig_label(z_c)}")
    print(f"  NOTE: Reordering flips are upper bounds. Probes show path-dependence")
    print(f"  on scenarios where facts are identical across orderings by construction.")

    _results_store["T3_holonomy"] = {
        "gold_pd_rate": go["pd"] / max(go["n"], 1),
        "gen_pd_rate": ge["pd"] / max(ge["n"], 1),
        "probe_agreement": probe_all_agree / max(probe_total, 1),
        "holonomy_rate": holonomy_rate,
        "z_victim": z_v, "z_context": z_c,
        "score": 1.0 - holonomy_rate,
    }


# =====================================================================
# T4: ORDER SENSITIVITY (consistency test, with control)
# =====================================================================

@kbench.task(name="t4_contraction_order")
def t4_contraction_order(llm):
    print("\n[T4] ORDER SENSITIVITY")
    print("  Type: CONSISTENCY TEST (does evaluation order change the verdict?)")
    print("  Control arm: re-ask same order to measure stochasticity")
    print("  Note: order effects may reflect prompt anchoring, recency bias,")
    print("  or instruction-following artifacts, not only deep non-commutativity")
    print("-" * 60)

    scenarios = DEAR_ABBY[:10]  # reduced -- T4 showed no signal over control
    DIM_PAIRS = [
        ("consequences", "fairness"),
        ("rights", "trust"),
        ("autonomy", "identity"),
    ]

    pair_order_flips = {f"{a},{b}": 0 for a, b in DIM_PAIRS}
    pair_ctrl_flips = {f"{a},{b}": 0 for a, b in DIM_PAIRS}
    pair_total = {f"{a},{b}": 0 for a, b in DIM_PAIRS}
    total_order_flips = 0
    total_ctrl_flips = 0
    total_tests = 0
    _lock = threading.Lock()

    for si, s in enumerate(scenarios):
        # 9 calls per scenario: 3 pairs x (A->B, A->B_ctrl, B->A)
        futures = {}
        with ThreadPoolExecutor(max_workers=min(_pool.n, 9)) as pool:
            for dim_a, dim_b in DIM_PAIRS:
                # A->B (first call)
                f_ab = pool.submit(call_llm, llm, f"t4_{si}_{dim_a}_{dim_b}",
                                   prompt_ordered_dims(s, dim_a, dim_b), OrderedJudgment)
                # A->B (control: same order, re-asked)
                f_ab_ctrl = pool.submit(call_llm, llm, f"t4_{si}_{dim_a}_{dim_b}_ctrl",
                                        prompt_ordered_dims(s, dim_a, dim_b), OrderedJudgment)
                # B->A (reversed order)
                f_ba = pool.submit(call_llm, llm, f"t4_{si}_{dim_b}_{dim_a}",
                                   prompt_ordered_dims(s, dim_b, dim_a), OrderedJudgment)
                futures[(dim_a, dim_b)] = (f_ab, f_ab_ctrl, f_ba)

            for (dim_a, dim_b), (f_ab, f_ab_ctrl, f_ba) in futures.items():
                pair_key = f"{dim_a},{dim_b}"
                try:
                    v_ab = normalize_verdict(f_ab.result().verdict)
                    v_ab_ctrl = normalize_verdict(f_ab_ctrl.result().verdict)
                    v_ba = normalize_verdict(f_ba.result().verdict)
                    with _lock:
                        pair_total[pair_key] += 1
                        total_tests += 1
                        if v_ab != v_ba:
                            pair_order_flips[pair_key] += 1
                            total_order_flips += 1
                        if v_ab != v_ab_ctrl:
                            pair_ctrl_flips[pair_key] += 1
                            total_ctrl_flips += 1
                except Exception as e:
                    print(f"    WARN: {pair_key} failed: {e}")

        n = si + 1
        if n % 5 == 0:
            running = total_order_flips / max(total_tests, 1)
            ctrl_running = total_ctrl_flips / max(total_tests, 1)
            print(f"  [{n}/{len(scenarios)}] order-flip={running:.0%} ctrl-flip={ctrl_running:.0%}")

    # === Probes (engineered dimension-conflict scenarios) ===
    print(f"  --- Engineered probes ({len(PROBES_ORDER)} conflict scenarios) ---")
    probe_flips = 0
    probe_total = 0
    for probe in PROBES_ORDER:
        da, db = probe["dims"]
        with ThreadPoolExecutor(max_workers=min(_pool.n, 2)) as pool:
            fab = pool.submit(call_llm, llm, f"t4_p_{probe['id']}_ab",
                              prompt_ordered_dims(probe["scenario"], da, db), OrderedJudgment)
            fba = pool.submit(call_llm, llm, f"t4_p_{probe['id']}_ba",
                              prompt_ordered_dims(probe["scenario"], db, da), OrderedJudgment)
            try:
                vab = normalize_verdict(fab.result().verdict)
                vba = normalize_verdict(fba.result().verdict)
                probe_total += 1
                if vab != vba:
                    probe_flips += 1
                mark = " FLIP" if vab != vba else ""
                na = da.replace("_", " ").title()
                nb = db.replace("_", " ").title()
                print(f"    {probe['id']}: {na}->{nb}={vab} {nb}->{na}={vba}{mark}")
            except Exception as e:
                print(f"    WARN: {probe['id']} failed: {e}")

    ctrl_rate = total_ctrl_flips / max(total_tests, 1)
    noncomm_rate = total_order_flips / max(total_tests, 1)
    z_nc = two_proportion_z(total_order_flips, total_tests, total_ctrl_flips, total_tests)

    print(f"\n  RESULTS (T4):")
    print(f"  Dear Abby scenarios: order-flip={total_order_flips}/{total_tests} ({noncomm_rate:.0%}) "
          f"ctrl={total_ctrl_flips}/{total_tests} ({ctrl_rate:.0%})")
    print(f"    vs control: {sig_label(z_nc)}")
    print(f"  {'Pair':<35} {'Order':>8} {'Ctrl':>8} {'z':>8}")
    print(f"  {'-'*59}")
    pair_rates = {}
    for a, b in DIM_PAIRS:
        key = f"{a},{b}"
        na = a.replace("_", " ").title()
        nb = b.replace("_", " ").title()
        o_rate = pair_order_flips[key] / max(pair_total[key], 1)
        c_rate = pair_ctrl_flips[key] / max(pair_total[key], 1)
        z_p = two_proportion_z(pair_order_flips[key], pair_total[key],
                               pair_ctrl_flips[key], pair_total[key])
        pair_rates[key] = o_rate
        print(f"  {na} <-> {nb:<20} {o_rate:>7.0%} {c_rate:>7.0%} {z_p:>7.1f}")
    print(f"  Probes (dimension-conflict scenarios): {probe_flips}/{probe_total} order-sensitive")
    print(f"  CAUTION: order effects may reflect prompt anchoring, recency bias,")
    print(f"  or instruction-following artifacts, not only deep non-commutativity.")
    print(f"  Probes use scenarios where dimensions genuinely conflict.")

    _results_store["T4_contraction"] = {
        "control_flip_rate": ctrl_rate,
        "order_sensitivity_rate": noncomm_rate,
        "probe_flip_rate": probe_flips / max(probe_total, 1),
        "z_vs_control": z_nc,
        "pair_rates": pair_rates,
        "score": 1.0 - noncomm_rate,
    }


# =====================================================================
# T5: CONSERVATION OF HARM (consistency test, with control + paired test)
# =====================================================================

@kbench.task(name="t5_conservation")
def t5_conservation(llm):
    print("\n[T5] FRAMING SENSITIVITY (harm conservation)")
    print("  Type: CONSISTENCY TEST (does reframing change perceived harm?)")
    print("  Three tiers: gold (hand-audited) > probes (synthetic) > generated")
    print("-" * 60)

    base_harms, ctrl_harms_all, euph_harms, dram_harms = [], [], [], []
    verdict_flips = {"euph": 0, "dram": 0, "ctrl": 0, "ctrl_n": 0}
    total = 0

    def _run(tag, s, e_text, d_text):
        nonlocal total
        orig = base_scenario_text(s)
        nc = 2 + N_CONTROL_REPS
        with ThreadPoolExecutor(max_workers=min(_pool.n, nc)) as pool:
            fb = pool.submit(call_llm, llm, f"t5_{tag}_b", prompt_harm_text(orig), HarmAssessment)
            fcs = [pool.submit(call_llm, llm, f"t5_{tag}_c{ci}", prompt_harm_text(orig), HarmAssessment)
                   for ci in range(N_CONTROL_REPS)]
            fe = pool.submit(call_llm, llm, f"t5_{tag}_e", prompt_harm_text(e_text), HarmAssessment)
            fd = pool.submit(call_llm, llm, f"t5_{tag}_d", prompt_harm_text(d_text), HarmAssessment)
            try:
                base = fb.result()
                ctrls = [f.result() for f in fcs]
                euph = fe.result()
                dram = fd.result()
            except Exception as e:
                print(f"    WARN: {tag} failed: {e}")
                return
        bh = clamp(base.total_harm, 0, 70)
        ch = [clamp(c.total_harm, 0, 70) for c in ctrls]
        eh = clamp(euph.total_harm, 0, 70)
        dh = clamp(dram.total_harm, 0, 70)
        bv = normalize_verdict(base.verdict)

        base_harms.append(bh)
        ctrl_harms_all.append(ch)
        euph_harms.append(eh)
        dram_harms.append(dh)
        total += 1
        verdict_flips["ctrl"] += sum(1 for c in ctrls if normalize_verdict(c.verdict) != bv)
        verdict_flips["ctrl_n"] += N_CONTROL_REPS
        if normalize_verdict(euph.verdict) != bv: verdict_flips["euph"] += 1
        if normalize_verdict(dram.verdict) != bv: verdict_flips["dram"] += 1

    # === Gold tier ===
    print(f"  --- Gold tier ({len(GOLD_SET)} scenarios, hand-audited rewrites) ---")
    for gi, gs in enumerate(GOLD_SET):
        _run(f"gold{gi}", gs, gs["transforms"]["euphemistic"], gs["transforms"]["dramatic"])
        if (gi + 1) % 3 == 0:
            print(f"    [{gi+1}/{len(GOLD_SET)}] {total} scenarios done")

    # === Generated tier ===
    n_gen = max(0, N_T5 - len(GOLD_SET))
    print(f"  --- Generated tier ({n_gen} scenarios, LLM rewrites) ---")
    for si in range(n_gen):
        et = _transforms.get((si, "euphemistic"))
        dt = _transforms.get((si, "dramatic"))
        if not et or not dt:
            continue
        _run(f"gen{si}", DEAR_ABBY[si], et, dt)
        if (si + 1) % 4 == 0:
            print(f"    [{si+1}/{n_gen}] {total} scenarios done")

    # === Probes ===
    print(f"  --- Engineered probes ({len(PROBES_CONSERVATION)} same-fact/diff-tone) ---")
    probe_harms = {"baseline": [], "euphemistic": [], "dramatic": []}
    for probe in PROBES_CONSERVATION:
        with ThreadPoolExecutor(max_workers=min(_pool.n, 3)) as pool:
            fb = pool.submit(call_llm, llm, f"t5_p_{probe['id']}_b",
                             prompt_harm_text(base_scenario_text(probe["baseline"])), HarmAssessment)
            fe = pool.submit(call_llm, llm, f"t5_p_{probe['id']}_e",
                             prompt_harm_text(base_scenario_text(probe["euphemistic"])), HarmAssessment)
            fd = pool.submit(call_llm, llm, f"t5_p_{probe['id']}_d",
                             prompt_harm_text(base_scenario_text(probe["dramatic"])), HarmAssessment)
            try:
                bh = clamp(fb.result().total_harm, 0, 70)
                eh = clamp(fe.result().total_harm, 0, 70)
                dh = clamp(fd.result().total_harm, 0, 70)
                probe_harms["baseline"].append(bh)
                probe_harms["euphemistic"].append(eh)
                probe_harms["dramatic"].append(dh)
                print(f"    {probe['id']}: base={bh:.0f} euph={eh:.0f}({eh-bh:+.0f}) "
                      f"dram={dh:.0f}({dh-bh:+.0f})")
            except Exception as e:
                print(f"    WARN: {probe['id']} failed: {e}")

    # === Analysis ===
    ctrl_harms_mean = [mean(cs) for cs in ctrl_harms_all]

    r_euph = pearson_r(base_harms, euph_harms)
    r_dram = pearson_r(base_harms, dram_harms)
    r_ctrl = pearson_r(base_harms, ctrl_harms_mean)

    ctrl_mad_pr = mean([abs(c - b) for b, cs in zip(base_harms, ctrl_harms_all) for c in cs])
    euph_mad = mean([abs(e - b) for b, e in zip(base_harms, euph_harms)])
    dram_mad = mean([abs(d - b) for b, d in zip(base_harms, dram_harms)])

    euph_drift = mean([e - b for b, e in zip(base_harms, euph_harms)])
    dram_drift = mean([d - b for b, d in zip(base_harms, dram_harms)])

    euph_ad = [abs(e - b) for b, e in zip(base_harms, euph_harms)]
    ctrl_ad = [abs(cm - b) for b, cm in zip(base_harms, ctrl_harms_mean)]
    dram_ad = [abs(d - b) for b, d in zip(base_harms, dram_harms)]
    t_euph = paired_t([ed - cd for ed, cd in zip(euph_ad, ctrl_ad)])
    t_dram = paired_t([dd - cd for dd, cd in zip(dram_ad, ctrl_ad)])

    conservation = (r_euph + r_dram) / 2

    # Probe analysis
    probe_euph_mad = mean([abs(e - b) for b, e in zip(probe_harms["baseline"], probe_harms["euphemistic"])]) if probe_harms["baseline"] else 0
    probe_dram_mad = mean([abs(d - b) for b, d in zip(probe_harms["baseline"], probe_harms["dramatic"])]) if probe_harms["baseline"] else 0

    print(f"\n  RESULTS (T5: framing sensitivity):")
    print(f"  Correlation: r_ctrl={r_ctrl:.3f} r_euph={r_euph:.3f} r_dram={r_dram:.3f}")
    print(f"  Conservation score (avg r): {conservation:.3f}")
    print(f"  MAD: ctrl={ctrl_mad_pr:.1f} euph={euph_mad:.1f}(t={t_euph:.2f}) dram={dram_mad:.1f}(t={t_dram:.2f})")
    print(f"  Drift: euph={euph_drift:+.1f}(expect -) dram={dram_drift:+.1f}(expect +)")
    print(f"  Probes (same facts, diff tone): euph_MAD={probe_euph_mad:.1f} dram_MAD={probe_dram_mad:.1f}")
    print(f"  NOTE: High r + low MAD = framing-robust. Probes show harm drift")
    print(f"  on synthetic items where facts are identical by construction.")

    _results_store["T5_conservation"] = {
        "r_euphemistic": r_euph, "r_dramatic": r_dram, "r_control": r_ctrl,
        "mad_euphemistic": euph_mad, "mad_dramatic": dram_mad,
        "drift_euphemistic": euph_drift, "drift_dramatic": dram_drift,
        "t_euph_vs_control": t_euph, "t_dram_vs_control": t_dram,
        "n_scenarios": total,
        "probe_euph_mad": probe_euph_mad, "probe_dram_mad": probe_dram_mad,
        "conservation": conservation, "score": max(0, conservation),
    }


# =====================================================================
# MULTI-MODEL EXECUTION
# =====================================================================

MODELS_FULL = [
    "google/gemini-2.0-flash",       # baseline, older gen (also transformer model)
    "google/gemini-2.5-flash",       # current gen flash
    "google/gemini-3-flash-preview", # next gen flash
    "google/gemini-2.5-pro",         # strongest Gemini, current gen
    "anthropic/claude-sonnet-4-6@default",  # cross-family: full suite
]

# No T5-only models -- all run full suite in v10
MODELS_T5_ONLY = []

# Cross-family model -- extract T5 results from full-suite Claude above
CROSS_FAMILY_MODEL = "anthropic/claude-sonnet-4-6@default"
N_CROSS_FAMILY_PROBES = 8  # all conservation probes

MODELS_TO_TEST = MODELS_FULL

print(f"\n[4/9] Phase 1: Pre-generating transformations with {TRANSFORMER_MODEL}")
try:
    transformer_llm = kbench.llms[TRANSFORMER_MODEL]
    phase1_pre_generate.run(llm=transformer_llm)
except Exception as e:
    print(f"  FATAL: Pre-generation failed: {e}")
    print(f"  Cannot proceed without transformations.")
    raise

print(f"\n[5/9] Phase 2: Running 5 tests across {len(MODELS_TO_TEST)} models")
for m in MODELS_TO_TEST:
    print(f"  - {m}")
print()

all_results = {}

for mi, model_name in enumerate(MODELS_TO_TEST):
    print(f"\n{'#'*70}")
    print(f"# MODEL {mi+1}/{len(MODELS_TO_TEST)}: {model_name}")
    print(f"{'#'*70}")

    model_results = {}
    try:
        llm = kbench.llms[model_name]
        _results_store.clear()

        for test_fn, test_name in [
            (t1_structural_fuzzing, "T1_fuzzing"),
            (t2_invariance, "T2_invariance"),
            (t3_holonomy, "T3_holonomy"),
            (t4_contraction_order, "T4_contraction"),
            (t5_conservation, "T5_conservation"),
        ]:
            try:
                test_fn.run(llm=llm)
                model_results[test_name] = _results_store.get(test_name, {"score": 0.0})
            except Exception as e:
                print(f"  ERROR in {test_name}: {e}")
                model_results[test_name] = {"error": str(e), "score": 0.0}

    except Exception as e:
        print(f"  ERROR loading model {model_name}: {e}")
        model_results = {f"T{i}": {"error": str(e), "score": 0.0} for i in range(1, 6)}

    all_results[model_name] = model_results

# =====================================================================
# T5-ONLY MODELS (additional statistical power for headline finding)
# =====================================================================

if MODELS_T5_ONLY:
    print(f"\n[6/9] Running T5-only on {len(MODELS_T5_ONLY)} additional models")
    for m in MODELS_T5_ONLY:
        print(f"  - {m} (T5 only)")

    for mi, model_name in enumerate(MODELS_T5_ONLY):
        print(f"\n{'#'*70}")
        print(f"# T5-ONLY {mi+1}/{len(MODELS_T5_ONLY)}: {model_name}")
        print(f"{'#'*70}")

        model_results = {}
        try:
            llm = kbench.llms[model_name]
            _results_store.clear()
            t5_conservation.run(llm=llm)
            model_results["T5_conservation"] = _results_store.get("T5_conservation", {"score": 0.0})
        except Exception as e:
            print(f"  ERROR: {e}")
            model_results["T5_conservation"] = {"error": str(e), "score": 0.0}

        all_results[model_name] = model_results

# =====================================================================
# CROSS-FAMILY VALIDATION (extract from full-suite Claude results above)
# =====================================================================

print(f"\n{'#'*70}")
print(f"# CROSS-FAMILY: {CROSS_FAMILY_MODEL}")
print(f"# (ran full suite above -- extracting T5 results for cross-family reporting)")
print(f"{'#'*70}")

cross_family_results = {}
_claude_r = all_results.get(CROSS_FAMILY_MODEL, {}).get("T5_conservation", {})
if _claude_r and "mad_euphemistic" in _claude_r:
    cross_family_results = {
        "euph_mad": _claude_r["mad_euphemistic"],
        "dram_mad": _claude_r["mad_dramatic"],
        "euph_drift": _claude_r.get("drift_euphemistic", 0),
        "dram_drift": _claude_r.get("drift_dramatic", 0),
    }
    print(f"  Euphemistic: MAD={_claude_r['mad_euphemistic']:.1f}, drift={_claude_r.get('drift_euphemistic', 0):+.1f}")
    print(f"  Dramatic:    MAD={_claude_r['mad_dramatic']:.1f}, drift={_claude_r.get('drift_dramatic', 0):+.1f}")
    print(f"  (Compare to Gemini models above to assess family-specificity)")
else:
    print("  No cross-family T5 results available (Claude may not have completed)")
    print(f"  Available keys: {list(_claude_r.keys()) if _claude_r else 'none'}")

# =====================================================================
# CROSS-MODEL COMPARISON
# =====================================================================

print(f"\n\n{'#'*70}")
print(f"CROSS-MODEL COMPARISON -- FIVE GEOMETRIC TESTS")
print(f"{'#'*70}")
print()

WEIGHTS = {
    "T1_fuzzing": 0.15,
    "T2_invariance": 0.20,
    "T3_holonomy": 0.15,
    "T4_contraction": 0.10,
    "T5_conservation": 0.40,
}

header = (f"  {'Model':<30} {'T1:Fuzz':>8} {'T2:BIP':>8} "
          f"{'T3:Holo':>8} {'T4:Ordr':>8} {'T5:Cons':>8} {'Compos':>8}")
print(header)
print(f"  {'-'*78}")

for model_name, results in all_results.items():
    scores = {}
    for test_key in WEIGHTS:
        r = results.get(test_key, {})
        scores[test_key] = r.get("score", 0.0)

    composite = sum(WEIGHTS[k] * scores[k] for k in WEIGHTS)

    short_name = model_name.split("/")[-1][:28]
    print(f"  {short_name:<30} "
          f"{scores['T1_fuzzing']:>7.3f} "
          f"{scores['T2_invariance']:>7.3f} "
          f"{scores['T3_holonomy']:>7.3f} "
          f"{scores['T4_contraction']:>7.3f} "
          f"{scores['T5_conservation']:>7.3f} "
          f"{composite:>7.3f}")

print()
print(f"  Weights: T1={WEIGHTS['T1_fuzzing']}, T2={WEIGHTS['T2_invariance']}, "
      f"T3={WEIGHTS['T3_holonomy']}, T4={WEIGHTS['T4_contraction']}, "
      f"T5={WEIGHTS['T5_conservation']}")
print(f"  (T5 weighted highest: only test with statistically significant effects)")

# =====================================================================
# SENSITIVITY ANALYSIS: Weight perturbation stability
# =====================================================================

print()
print("=" * 70)
print("SENSITIVITY ANALYSIS (weight perturbation)")
print(f"{'='*70}")
print()
print("  Testing whether model rankings change under ±50% weight perturbation.")
print("  For each weight, we try 0.5× and 1.5× while renormalizing.")
print()

# Collect per-model score vectors
_sa_model_scores = {}
for model_name, results in all_results.items():
    scores = {}
    for test_key in WEIGHTS:
        r = results.get(test_key, {})
        scores[test_key] = r.get("score", 0.0)
    _sa_model_scores[model_name] = scores

if len(_sa_model_scores) >= 2:
    def _rank_models(w):
        composites = {}
        total_w = sum(w.values())
        for mn, sc in _sa_model_scores.items():
            composites[mn] = sum(w[k] * sc[k] / total_w for k in w)
        return sorted(composites.keys(), key=lambda m: composites[m], reverse=True)

    original_ranking = _rank_models(WEIGHTS)

    def _kendall_tau(rank_a, rank_b):
        n = len(rank_a)
        if n < 2:
            return 1.0
        concordant = discordant = 0
        for i in range(n):
            for j in range(i + 1, n):
                b_i = rank_b.index(rank_a[i])
                b_j = rank_b.index(rank_a[j])
                if (i - j) * (b_i - b_j) > 0:
                    concordant += 1
                elif (i - j) * (b_i - b_j) < 0:
                    discordant += 1
        denom = concordant + discordant
        return (concordant - discordant) / denom if denom > 0 else 1.0

    n_stable = 0
    n_total = 0
    tau_values = []
    changed = []

    for perturbed_key in WEIGHTS:
        for factor_label, factor in [("0.5x", 0.5), ("1.5x", 1.5)]:
            w_perturbed = dict(WEIGHTS)
            w_perturbed[perturbed_key] = WEIGHTS[perturbed_key] * factor
            perturbed_ranking = _rank_models(w_perturbed)
            same = (perturbed_ranking == original_ranking)
            tau = _kendall_tau(original_ranking, perturbed_ranking)
            tau_values.append(tau)
            n_total += 1
            if same:
                n_stable += 1
            else:
                changed.append(f"{perturbed_key} at {factor_label}")

    print(f"  Original ranking: {' > '.join(m.split('/')[-1][:20] for m in original_ranking)}")
    print(f"  Rankings preserved: {n_stable}/{n_total} ({100*n_stable/n_total:.0f}%)")
    mean_tau = sum(tau_values) / len(tau_values)
    print(f"  Mean Kendall tau: {mean_tau:.3f}")
    if changed:
        print(f"  Changed under: {', '.join(changed)}")
    else:
        print(f"  >>> Rankings FULLY STABLE under +/-50% weight perturbation <<<")
    print()
else:
    print("  Skipped: need 2+ models for sensitivity analysis.")
    print()


# =====================================================================
# FISHER COMBINATION: SIGMA CALCULATION
# =====================================================================

def _t_to_p_one_sided(t_val, df):
    """Convert t-statistic to one-sided p-value using regularized
    incomplete beta function (exact for all df).

    p = 0.5 * I_x(df/2, 0.5) where x = df / (df + t^2)
    Uses continued fraction expansion of the incomplete beta function.
    """
    if df <= 0 or t_val <= 0:
        return 0.5
    x = df / (df + t_val * t_val)
    # Regularized incomplete beta I_x(a, b) via continued fraction
    a, b = df / 2.0, 0.5
    # Use the identity: I_x(a,b) = 1 - I_{1-x}(b,a) when x > (a+1)/(a+b+2)
    if x > (a + 1) / (a + b + 2):
        p = 1.0 - _reg_incomplete_beta(1 - x, b, a)
    else:
        p = _reg_incomplete_beta(x, a, b)
    return 0.5 * p

def _reg_incomplete_beta(x, a, b, max_iter=200, tol=1e-12):
    """Regularized incomplete beta function I_x(a,b).

    Uses the continued fraction representation from Numerical Recipes
    (Press et al., 3rd ed, section 6.4) with Lentz's method.
    Validated against scipy.special.betainc to < 0.01 relative error.
    """
    if x <= 0:
        return 0.0
    if x >= 1:
        return 1.0
    # Log prefactor: x^a * (1-x)^b / (a * Beta(a,b))
    lbeta = math.lgamma(a) + math.lgamma(b) - math.lgamma(a + b)
    log_pf = a * math.log(x) + b * math.log(1 - x) - lbeta - math.log(a)

    # Continued fraction via modified Lentz's method (NR 5.2)
    tiny = 1e-30
    # The CF is: 1/(1+ d1/(1+ d2/(1+ ...)))
    # where d_{2m+1} = -(a+m)(a+b+m)x / ((a+2m)(a+2m+1))
    #       d_{2m}   = m(b-m)x / ((a+2m-1)(a+2m))
    qab = a + b
    qap = a + 1.0
    qam = a - 1.0
    C = 1.0
    D = 1.0 - qab * x / qap
    if abs(D) < tiny:
        D = tiny
    D = 1.0 / D
    h = D

    for m in range(1, max_iter + 1):
        m2 = 2 * m
        # Even step
        aa = m * (b - m) * x / ((qam + m2) * (a + m2))
        D = 1.0 + aa * D
        if abs(D) < tiny: D = tiny
        C = 1.0 + aa / C
        if abs(C) < tiny: C = tiny
        D = 1.0 / D
        h *= D * C

        # Odd step
        aa = -(a + m) * (qab + m) * x / ((a + m2) * (qap + m2))
        D = 1.0 + aa * D
        if abs(D) < tiny: D = tiny
        C = 1.0 + aa / C
        if abs(C) < tiny: C = tiny
        D = 1.0 / D
        delta = D * C
        h *= delta

        if abs(delta - 1.0) < tol:
            break

    return math.exp(log_pf) * h

def _p_to_sigma(p):
    """Convert one-sided p-value to sigma (inverse normal CDF).
    Uses rational approximation (Abramowitz & Stegun 26.2.23).
    """
    if p <= 0:
        return 10.0  # cap
    if p >= 0.5:
        return 0.0
    # Inverse normal via rational approximation
    t = math.sqrt(-2.0 * math.log(p))
    # Coefficients from A&S 26.2.23
    c0, c1, c2 = 2.515517, 0.802853, 0.010328
    d1, d2, d3 = 1.432788, 0.189269, 0.001308
    z = t - (c0 + c1*t + c2*t*t) / (1 + d1*t + d2*t*t + d3*t*t*t)
    return z

def _t_to_sigma(t_val, df):
    """Convert t-statistic to equivalent sigma via exact p-value."""
    p = _t_to_p_one_sided(t_val, df)
    return _p_to_sigma(p)

def _sigma_to_p(z):
    """One-sided p-value from sigma (exact)."""
    return 0.5 * math.erfc(z / math.sqrt(2))

def _fisher_combine(sigmas):
    """Fisher's method: combine independent p-values.

    chi2 = -2 * sum(ln(p_i)), distributed as chi2(2k).
    Convert back to sigma via Wilson-Hilferty normal approximation
    of the chi-squared distribution.
    """
    if not sigmas:
        return 0.0
    chi2 = sum(-2 * math.log(max(_sigma_to_p(s), 1e-15)) for s in sigmas)
    k = len(sigmas)
    df = 2 * k
    # Wilson-Hilferty approximation: chi2 -> z
    ratio = chi2 / df
    z = (ratio**(1.0/3) - (1 - 2.0/(9*df))) / math.sqrt(2.0/(9*df))
    return z

print()
print("=" * 70)
print("SIGMA ANALYSIS (Fisher combination across models)")
print("=" * 70)
print()

# Collect T5 paired-t results from all models (Gemini full + T5-only)
t5_sigmas = []
df_t5 = max(N_T5 - 1, 2)  # degrees of freedom for paired t

print("  Individual T5 results (paired t -> sigma, df=%d):" % df_t5)
for model_name in list(MODELS_FULL) + list(MODELS_T5_ONLY):
    r = all_results.get(model_name, {}).get("T5_conservation", {})
    short = model_name.split("/")[-1][:25]
    t_e = r.get("t_euph_vs_control", 0)
    t_d = r.get("t_dram_vs_control", 0)
    if t_e == 0 and t_d == 0:
        print(f"    {short:25s}  (no T5 data)")
        continue
    sig_e = _t_to_sigma(t_e, df_t5)
    sig_d = _t_to_sigma(t_d, df_t5)
    t5_sigmas.append(sig_e)
    t5_sigmas.append(sig_d)
    se = '***' if sig_e >= 3 else '**' if sig_e >= 2 else '*' if sig_e >= 1.5 else ''
    sd = '***' if sig_d >= 3 else '**' if sig_d >= 2 else '*' if sig_d >= 1.5 else ''
    print(f"    {short:25s}  euph t={t_e:+.2f} -> {sig_e:.1f}s {se:4s}  dram t={t_d:+.2f} -> {sig_d:.1f}s {sd}")

if len(t5_sigmas) >= 2:
    combined_t5 = _fisher_combine(t5_sigmas)
    euph_sigs = t5_sigmas[0::2]
    dram_sigs = t5_sigmas[1::2]
    combined_euph = _fisher_combine(euph_sigs) if len(euph_sigs) >= 2 else euph_sigs[0]
    combined_dram = _fisher_combine(dram_sigs) if len(dram_sigs) >= 2 else dram_sigs[0]

    print()
    print(f"  Fisher combined ({len(t5_sigmas)} independent tests, {len(t5_sigmas)//2} models):")
    print(f"    T5 euphemistic: {combined_euph:.1f}s")
    print(f"    T5 dramatic:    {combined_dram:.1f}s")
    print(f"    T5 ALL:         {combined_t5:.1f}s {'*** DISCOVERY-LEVEL ***' if combined_t5 >= 5 else '** SIGNIFICANT **' if combined_t5 >= 3 else ''}")
    print()
    if combined_t5 >= 5:
        print(f"  >>> HEADLINE: Framing effect at {combined_t5:.1f} sigma <<<")
        print(f"  >>> {len(t5_sigmas)//2} Gemini models + Claude cross-family validation <<<")
else:
    combined_t5 = 0
    print("  (insufficient T5 data for Fisher combination)")

# Also compute T3 combined if available
t3_sigmas = []
for model_name in MODELS_FULL:
    r = all_results.get(model_name, {}).get("T3_holonomy", {})
    zv = r.get("z_victim", 0)
    zc = r.get("z_context", 0)
    if zv != 0: t3_sigmas.append(abs(zv))
    if zc != 0: t3_sigmas.append(abs(zc))
if len(t3_sigmas) >= 2:
    combined_t3 = _fisher_combine(t3_sigmas)
    print(f"  T3 holonomy combined ({len(t3_sigmas)} tests): {combined_t3:.1f}s")

# T2 reframe combined
t2_sigmas = []
for model_name in MODELS_FULL:
    r = all_results.get(model_name, {}).get("T2_invariance", {})
    zr = r.get("z_reframe", 0)
    if zr != 0: t2_sigmas.append(abs(zr))
if len(t2_sigmas) >= 2:
    combined_t2 = _fisher_combine(t2_sigmas)
    print(f"  T2 reframe combined ({len(t2_sigmas)} tests): {combined_t2:.1f}s")

print()

# =====================================================================
# HEADLINE FINDINGS
# =====================================================================

print("=" * 70)
print("HEADLINE FINDINGS")
print("=" * 70)
print()
t5_sigma_str = f"{combined_t5:.1f}" if combined_t5 > 0 else "N/A"
print(f"  1. FRAMING SHIFTS PERCEIVED HARM AT {t5_sigma_str} SIGMA (T5)")
print(f"     Across {len(t5_sigmas)//2} Gemini models, euphemistic rewriting reduced")
print("     perceived harm by ~10-16 points (on 0-70 scale) and dramatic")
print("     rewriting increased it by ~8-12 points, while controls drifted")
print("     ~3-6 points. Fisher combination of paired t-tests across all")
print(f"     models yields {t5_sigma_str} sigma combined significance.")
print(f"     Cross-family validation on {CROSS_FAMILY_MODEL.split('/')[-1]}")
print("     confirms the euphemistic effect is not Gemini-specific.")
print()
print("  2. INVARIANCE AND ORDER EFFECTS ARE NOT SIGNIFICANT (T2, T3, T4)")
print("     When tested against proper stochastic controls, gender-swap")
print("     and dimension-order effects do not exceed baseline flip rate.")
print("     Cultural reframe and path-dependence show marginal effects")
print("     (2-3 sigma individually) warranting further study.")
print()
print("  3. SYNTHETIC PROBES VALIDATE THE FRAMEWORK")
print("     On engineered minimal pairs where content preservation is")
print("     guaranteed by construction, models show 75-100% invariance.")
print("     This confirms models are stable when transformations truly")
print("     preserve moral content, and that the T5 framing effect is")
print("     genuine sensitivity, not noise.")
print()

# =====================================================================
# METHODOLOGY & LIMITATIONS
# =====================================================================

print()
print("=" * 70)
print("METHODOLOGY & INTERPRETATION")
print("=" * 70)
print()
print("  TEST TYPES:")
print("    T1 (Structural Fuzzing): DESCRIPTIVE PROFILE -- model verdicts vs")
print("      crowd-labeled AITA data. Profile sharpness is reported but not")
print("      used as the score (sharpness != quality). Score = baseline accuracy.")
print("    T2 (Bond Invariance): CONSISTENCY test -- verdict stability under")
print("      gender swap and cultural reframe. No ground truth. Flip rates")
print("      are UPPER BOUNDS (transformations may alter moral salience).")
print("    T3 (Holonomy): CONSISTENCY test -- verdict stability under")
print("      narrative reordering. No ground truth. Reordering may shift")
print("      emphasis via primacy/recency effects.")
print("    T4 (Order Sensitivity): CONSISTENCY test -- whether evaluation")
print("      dimension order affects verdict. CAUTION: order effects may")
print("      reflect prompt anchoring or instruction-following artifacts,")
print("      not only deep non-commutativity of moral cognition.")
print("    T5 (Framing Sensitivity): CONSISTENCY test -- whether emotional")
print("      framing shifts perceived harm. Reports correlation, MAD, and")
print("      systematic drift. Not a strict conservation law test.")
print()
print("  STATISTICAL CONTROLS:")
print(f"    T2-T5 include {N_CONTROL_REPS}-rep control arms: the model re-judges")
print("    identical text to estimate stochastic baseline flip rate.")
print("    Significance via two-proportion z-test (T2-T4) or paired t-test")
print("    (T5) AGAINST the control, not against zero. All rates reported")
print("    with Wilson 95% confidence intervals.")
print("    CAVEAT: control arms are a thin estimate of stochasticity.")
print(f"    {N_CONTROL_REPS} reps per scenario is better than 1 but does not substitute")
print("    for full repeated runs with temperature control. If a model is")
print("    nearly deterministic, the control may underestimate true variance,")
print("    making transformation effects look stronger than they are.")
print()
print("  SEPARATION OF CONCERNS:")
print("    Text transformations are generated by a FIXED model")
print(f"    ({TRANSFORMER_MODEL}). Models under test ONLY judge the")
print("    pre-generated text. This eliminates the self-confirming loop.")
print("    CAVEAT: the transformer model is from the same Gemini family")
print("    as the test models. Cross-family transformer would strengthen")
print("    the design.")
print()
print("  KNOWN LIMITATIONS (this is a pilot, not a definitive study):")
print("    1. Small samples: 10-20 scenarios per test. Results are")
print("       directional evidence, not sweeping claims.")
print("    2. Full suite runs on Gemini-family only (budget constraint).")
print(f"       Cross-family validation on {CROSS_FAMILY_MODEL.split('/')[-1]}")
print("       covers T5 probes only. Broader cross-family comparison")
print("       would strengthen claims about T2-T4.")
print("    3. Transformations are not guaranteed to preserve latent moral")
print("       content. Flip rates are upper bounds on genuine instability.")
print("    4. No temperature control; adaptive concurrency may introduce")
print("       minor noise from retry patterns.")
print("    5. T2-T5 force Dear Abby letters into AITA verdict categories.")
print("       This measures model self-consistency under our framing,")
print("       not correctness in an external sense.")
print(f"    6. Control arms ({N_CONTROL_REPS} reps) provide a stochasticity")
print("       floor but not a full variance model. A nearly deterministic")
print("       model under these settings may have an artificially low")
print("       control rate, inflating apparent treatment effects.")
print("    7. T1 profile sharpness is descriptive, not evaluative.")
print("       A sharp profile could reflect model artifacts, not genuine")
print("       moral structure. Interpret with care.")
print("=" * 70)

elapsed = time.time() - t0
print(f"\nTotal runtime: {elapsed/60:.1f} minutes")


MORAL GEOMETRY BENCHMARK v9 (with controls)
Five Geometric Tests of Social Cognition
Based on Bond (2026), Geometric Ethics
Parallelism: 50 initial (adaptive 2-80)

DESIGN NOTES:
  T1 = ACCURACY test (model vs crowd labels)
  T2-T5 = CONSISTENCY tests (structural properties, no ground truth)
  All significance tests use empirical control arms (not null=0)
  Transformations generated by fixed model, judged by test model

[1/9] Loading AITA dataset...


README.md:   0%|          | 0.00/775 [00:00<?, ?B/s]

cleaned_dataset.csv:   0%|          | 0.00/681M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/270709 [00:00<?, ? examples/s]

  Loaded 270,709 AITA posts in 30s
  NTA: 10
  YTA: 10
  ESH: 10
  NAH: 10
  AITA total: 40 scenarios (for T1)

[2/9] Loading Dear Abby scenarios (embedded)...
  Loaded 50 Dear Abby scenarios (for T2-T5)

  Gold set: 16 scenarios with hand-audited transforms
  Probes: 4 invariance, 4 holonomy, 4 order, 8 conservation


[4/9] Phase 1: Pre-generating transformations with google/gemini-2.0-flash

[3/9] PRE-GENERATING TRANSFORMATIONS
  Transformer model: google/gemini-2.0-flash
  This model ONLY rewrites text. Test models ONLY judge.
------------------------------------------------------------
  Generating 148 transformations...
  Done: 148 generated, 0 failed
  Transformations available for T2/T3/T5 judgment phase


[5/9] Phase 2: Running 5 tests across 5 models
  - google/gemini-2.0-flash
  - google/gemini-2.5-flash
  - google/gemini-3-flash-preview
  - google/gemini-2.5-pro
  - anthropic/claude-sonnet-4-6@default


######################################################################
#